Martin Torres

Architectural Engineering PhD Student | University of Colorado, Boulder

Dr. Wil Srubar's Living Materials Laboratory

mtorres2941@gmail.com

# Import

In [ ]:
import arviz as az
from itertools import product, combinations, permutations
import math
import matplotlib
import matplotlib.pyplot as plt
import mpltern
import numpy as np
import os
import pandas as pd
import pymc as pm
import random
import scipy
from scipy.stats import norm, gamma, dirichlet
import seaborn as sns
from tqdm import tqdm
import xarray as xr
import warnings

In [ ]:
import sys
sys.path.insert(0, 'src')

from funcs_kde2 import check_array, weighted_quantile, kl2, kl2_plot, weighted_bw, weighted_std, bw_dirichlet

from funcs_unit_conversion import hi2, hi3, lo2
from funcs_unit_conversion import m2_ft2, m3_ft3, m3_yd3, psi_mpa

from funcs_unit_conversion import area_units, m2, ft2
from funcs_unit_conversion import densityarea_units, kg_m2, kg_yd2, psf, kg_dm2, t_m2, g_m2, oz_yd2
from funcs_unit_conversion import density_units, kg_m3, kg_yd3, lb_ft3, kg_dm3, t_m3
from funcs_unit_conversion import emission_units, lbco2e, tco2e, kgco2e
from funcs_unit_conversion import energyemission_units, lbco2mwh, tco2mwh, kgco2mwh 
from funcs_unit_conversion import length_units, m1, ft, cm, mm, inch, km
from funcs_unit_conversion import pressure_units, psi, ksi, mpa, nmm2
from funcs_unit_conversion import therm_units, uval, rval, rsival
from funcs_unit_conversion import time_units, decade, year, day, hour, minute, second
from funcs_unit_conversion import volume_units, ft3, m3, yd3
from funcs_unit_conversion import weight_units, kgs, gs, lbs, tons, tonnes

from funcs_unit_conversion import area2m2, density2kgm2, density2kgm3, emission2kgco2e, emission2kgmwh, length2in, pressure2psi, therm2rval, time2year, vol2m3, weight2kgs
from funcs_unit_conversion import dict_unitconv, dict_unittype, consistent_units, str2valunit

In [ ]:
matplotlib.rcParams['figure.dpi'] = 1200

hix = r'$\mathrm{\mathsf{   ^x   }}$'
hi2 = r'$\mathrm{\mathsf{   ^2   }}$'
hi3 = r'$\mathrm{\mathsf{   ^3   }}$'
lo1 = r'$\mathrm{\mathsf{   _1   }}$'
lo2 = r'$\mathrm{\mathsf{   _2   }}$'
lo3 = r'$\mathrm{\mathsf{   _3   }}$'
locomma = r'$\mathrm{\mathsf{   _{,}   }}$'
lodata = r'$\mathrm{\mathsf{   _{data}   }}$'
loi = r'$\mathrm{\mathsf{   _{i}   }}$'
loia = r'$\mathrm{\mathsf{   _{IA}   }}$'
loiqr = r'$\mathrm{\mathsf{   _{IQR}   }}$'
looriginal = r'$\mathrm{\mathsf{  _{original}     }}$'
lophantom = r'$\mathrm{\mathsf{  _{phantom}     }}$'
lopi = r'$\mathrm{\mathsf{   _{Π}   }}$'
losim = r'$\mathrm{\mathsf{  _{\backsim}   }}$'
lotarget = r'$\mathrm{\mathsf{   _{target}   }}$'


## def kl2(data, xplot, group_constraints, BW_factors, evtargets, represented, nruns, progressbar)

In [ ]:
def kl2(data,
        xplot,
        group_constraints=None,
        BW_factors=None, 
        evtargets=None,
        represented=None,
        nruns=1000,
        progressbar=True
       ):
    # group_constraints=None
    # BW_factors=None
    # evtargets=None
    # represented=None
    # nruns=1000
    # progressbar=True
    """
    This function performs advanced KDE to a dataset, accommodating 
    multiple sources of uncertainty and conveying a series of possible 
    distributions as part of a Dirichlet Process.

    INPUTS:
        data                All data points considered for this analysis in array-like format
        xplot               This must be a user input so you know how to plot the resulting yplot. Use np.linspace(start, stop, num)
        group_constraints   This is a dictionary where the key is a tuple of indices and the value is the proportion of weight those indices should take up
        BW_factors          Bandwidth factors for each point in "data". Mean will be taken as one
        evtargets           The set of feasible targets for expected value of the resulting plot. These values are used to create a KDE plot, which is sampled from for each iteration's target expected value
        represented         What percentage of the overall data is represented by this dataset. Weight represented by new, random values
        nruns               How many probability density functions are produced. Recommended minimum 1000, which is the autofill value
        progressbar         Boolean that dictates whether messages are printed or not. This should be True unless you're running many separate simulations in a row.

    OUTPUTS:
        xplot               Array of x values produced with np.linspace(). May have been resized to accommodate high values
        yplot               Array of (xplot * nruns) representing all resulting probability density functions
        dict_analytics      Dictionary of some results, including all_x, all_w, and all_ev
    """
    ####################################################################################
    ######################  CHECK INPUT DATA ###########################################
    ####################################################################################
    X = np.array(data).flatten()
    lst1 = []
    lst2 = []
    messages = []
    #check group_constraints and represented
    if group_constraints is None:
        if represented is None:
            represented = 0.6
            messages.append('represented has defaulted to 0.6, indicating that this dataset represents 60% of all values')
        elif represented > 1 or represented < 0:
            raise ValueError("The variable 'represented' must be between 0 and 1")        
        group_constraints = {
            tuple(np.arange(len(X))): represented,
        }
        messages.append('No group constraints were provided')
    else:
        # check for repeating indices
        gc_inds = [item for lst in group_constraints.keys() for item in lst]
        repeats = set([x for x in gc_inds if gc_inds.count(x) > 1])
        if len(repeats)>0:
            raise ValueError(f'The following indices in group_constraints are repeating: {repeats}')

        # check for out of bounds indices
        oob = [ele for ele in gc_inds if ele>=len(data)]
        if len(oob)>0:
            raise ValueError(f'group_constraints indices out of bounds. Dataset length: {len(X)}. Indices: {oob}')

        # check for unused indices
        ind_remainder = [ele for ele in np.arange(len(X)) if ele not in gc_inds]
        gc_sum = sum(group_constraints.values())

        # check if represented and group_constraints agree - allocate weight as needed.
        if len(ind_remainder) > 0: #there are remaining indices
            if represented is None:
                raise ValueError(f'Indices were left out of group_constraints and represented was not given. One of these must be corrected. Indices include {ind_remainder}')
            elif represented > 1 or represented < 0:
                raise ValueError("The variable 'represented' must be between 0 and 1")    
            elif represented < gc_sum:
                raise ValueError(f'represented is less than the sum of group_constraints, so there is no remaining weight to be allocated to unconstrained indices: {ind_remainder}')
            group_constraints[tuple(ind_remainder)] = represented-gc_sum

        else: #there are no more remaining indices
            if represented is None:
                represented = gc_sum
                messages.append(f'represented defaulted to the sum of group_constraints: {gc_sum}')
            elif represented > 1 or represented < 0:
                raise ValueError("The variable 'represented' must be between 0 and 1")    
            elif represented != gc_sum:
                messages.append(f'represented does not equal the sum of group_constraints and all indices are accounted for, so represented is being changed from {represented} to {gc_sum}')
                represented = gc_sum
    
    gc_sum = sum(group_constraints.values())
    if represented != gc_sum:
        raise ValueError(f'represented does not equal the sum of group constraints. represented={represented}, sum(gc)={gc_sum}')
    elif represented <= 0 or represented > 1.0:
        raise ValueError(f'represented must be greater than zero and less than or equal to one. Input was {represented}')
        
    # get average weight
    Wavg = np.zeros_like(X).astype(float)
    for inds, perc in group_constraints.items():
        for ind in inds:
            Wavg[ind] = perc/len(inds)

    #check BW_factors
    wstd = Wavg/sum(Wavg)
    base_ev = sum(X*wstd)/sum(wstd)
    # base_std = np.sqrt(sum(wstd*(X-sum(X*wstd)/sum(wstd))**2)/sum(wstd))    
    # iqr = weighted_quantile(X, wstd, 0.75, output='perc2val') - weighted_quantile(X, wstd, 0.25, output='perc2val')
    # bw_base = 0.9 * min([base_std, iqr/1.34]) * len(X)**-0.2
    bw_base = weighted_bw(X, wstd, bw_method='silverman')
    if BW_factors is None:
        # BW_factors = np.ones(len(X)+1)
        BW_factors = np.ones(len(X))
        messages.append('BW_factors has defaulted an array of ones so all bandwidths will be equal.')

    else:
        BW_factors = check_array(BW_factors, X, 'BW_factors')
        BW_factors = BW_factors / np.mean(BW_factors)
        # BW_factors = np.concatenate([BW_factors, [1]])

    #check evtargets
    if evtargets is None:
        evtargets = np.random.normal(base_ev, bw_base, nruns)
        messages.append(f'evtargets has defaulted to a normal distribution with mean = expected value, stdev = bandwidth calculated with the Silverman method')
    else:
        evtargets = np.array(evtargets).flatten()        

    #check xplot - must be custom input array, likely from np.linspace(low, high, number_of_steps)
    if len(xplot) == nruns:
        nruns += 1
        messages.append('nruns has been increased by 1 to avoid being the same integer as xplot. This is to avoid confusing dimensions of the results.')


    if progressbar:
        for message in messages:
            print(message)
            
            
    ####################################################################################
    ######################  FUNCTION STARTS HERE #######################################
    ####################################################################################

    #collect metadata
    n_added = 1
    W_all = np.zeros(shape=(nruns,len(X)+n_added))
    X_all = np.zeros(shape=(nruns,len(X)+n_added))
    EV_all = np.zeros_like(range(nruns))
    yplot = np.zeros_like(xplot, shape=(nruns,len(xplot)))
    xplot_bust = []
        
    #run Monte Carlo simulations
    for run in tqdm(range(nruns), desc='FOR LOOP PROGRESS', disable=not progressbar):
        # reset data
        X = np.array(data).flatten()
        Wavg = Wavg[:len(X)]
        BW_factors = BW_factors[:len(X)]
        
        # pull target
        target = evtargets[run % len(evtargets)]

        # randomly sample weights from Dirichlet distribution
        W = np.zeros_like(X).astype(float)
        for inds, perc in group_constraints.items():
            wnew = np.random.dirichlet(np.ones(len(inds)))*perc
            for ind, w in zip(inds, wnew):
                W[ind] = w.copy()

        # simulate phantom kernel
        EV = sum(X*W)/sum(W)
        if represented == 1:
            xnew = np.array(target).flatten()
            wnew = np.array(0).flatten()
        else:
            xnew = target + (target-EV)*(represented)/(1-represented)
            xnew = np.array(np.max([xnew, 0])).flatten()
            wnew = np.array(1-represented).flatten()

        # calculate bandwidth such that variance of the resulting plot >= variance of the original plot
        # without phantom kernel
        # BW1 = bw_dirichlet(X, Wavg, BW_factors, Wrun=W, bw_method='silverman')
        # BW1 = np.concatenate([BW1, [np.mean(BW1)]])
        # # with phantom kernel
        # X = np.concatenate([np.array(X).flatten(), xnew])
        # Wavg = np.concatenate([Wavg, [1-represented]])
        # BW_factors = np.concatenate([BW_factors, [1.]])
        # W = np.concatenate([np.array(W).flatten(), wnew])
        # BW2 = bw_dirichlet(X, Wavg, BW_factors, Wrun=W, bw_method='silverman')
        # BW = np.max([BW1, BW2], axis=0)
        
        # calculate bandwidth with baseline bandwidth
        BW_factors = np.concatenate([BW_factors, [1.]])
        BW = BW_factors*bw_base
        W = np.concatenate([np.array(W).flatten(), wnew])
        X = np.concatenate([np.array(X).flatten(), xnew])


        # re calculate bandwidth each time
        # bw_basenew = weighted_bw(X, W, bw_method='silverman')
        # BW = BW_factors*bw_basenew
        # W = np.concatenate([np.array(W).flatten(), wnew])
        # X = np.concatenate([np.array(X).flatten(), xnew])

        
        # record results        
        W_all[run] = W
        X_all[run] = X
        EV_all[run] = sum(X*W)/sum(W)

        #check to see if xplot captures everything
        hi = np.max(X + 3*BW)
        if max(xplot) < hi:
            xplot_bust.append(hi)
            # xplot = np.linspace(0,hi,len(xplot))
            # if progressbar:
                # print(f'The upper limit of xplot has been increased to accommodate the upper limit: {hi}')
                # print(f'Consider adjusting the upper limit of xplot. np.max(X + 3*BW) is {np.max(X + 3*BW)} and np.max(xplot) is {np.max(xplot)}')
        
        # create each PDF and ensure it's valid and sums to 1
        yi = norm.pdf(xplot.reshape(-1,1), X, BW)*W
        # yplot[run] = sum(yi.T)
        yplot[run] = yi.sum(axis=1)
        yplot[run] /= np.trapz(yplot[run], xplot)

    dict_analytics = {}
    dict_analytics['all_x'] = X_all
    dict_analytics['all_w'] = W_all
    dict_analytics['all_ev'] = EV_all
    if len(lst1) > 0:
        dict_analytics['lst1'] = lst1
    if len(lst2) > 0:
        dict_analytics['lst2'] = lst2
    if len(xplot_bust) > 0 and progressbar:
        print(f'X+3*BW was greater than max(xplot)={np.max(xplot)} for {len(xplot_bust)}/{nruns} runs, with a max of {np.max(xplot_bust)}')


    return xplot, yplot.T, dict_analytics

In [ ]:
%%time
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
bw_base = 0.9*np.std(X)*len(X)**-0.2
xplot = np.linspace(0, 50, 1_001)

mean = 0.75*np.mean(X)
std = 0.5*np.std(X)
rep = 0.5


MEAN = np.array([1/4, 2/4, 3/4, 4/4, 5/4])*np.mean(X)
STD = np.array([0/4, 1/4, 2/4, 3/4, 4/4])*np.std(X)
REP = np.arange(1,10)/10

evtargets = np.random.normal(mean, std, 1000)
xplot, yplot, dict_analytics = kl2(data=X, xplot=xplot, evtargets=evtargets, represented=rep, nruns=100, progressbar=False)


In [ ]:
fig, ax = plt.subplots(figsize=(6,2))
ax.plot(xplot, yplot, linewidth=0.1, color='black');
ax.set_title('making sure kl2 function works');

## def kl2_plot(xplot, yplot, ax, lo=0.25, hi=0.75, color='tab:red')

In [ ]:
def kl2_plot(xplot, yplot, ax, lo=0.25, hi=0.75, color='tab:red', include_area_iqr=True):
    """
    INPUTS: xplot, yplot (output from kl2), ax, lo=0.1, hi=0.9
    OUTPUTS: plot, no return
    """
    qlo = np.quantile(yplot, lo, axis=1)
    mean = np.mean(yplot.T, axis=0)
    mean /= scipy.integrate.trapezoid(mean, xplot)
    qhi = np.quantile(yplot, hi, axis=1)
    nruns = yplot.shape[0]

    # fig, ax = plt.subplots(1,1)
    #plot regular kde applied to data
    # sns.kdeplot(data=data, bw_method='silverman', label='Regular KDE', ax=ax)

    # plot iterations
    ax.plot(xplot, yplot, alpha=0.01, color='black');
    ax.plot(xplot, mean, color='gray', label=f"{yplot.shape[1]} iterations");

    # plot quartiles
    ax.plot(xplot, mean, color=color, label='Mean PDF');
    ax.plot(xplot, qlo, color=color, linestyle='--',
            label=f"{int(np.round(lo*100,0))}-{int(np.round(hi*100,0))}%");
    ax.plot(xplot, qhi, color=color, linestyle='--');
    ax.fill_between(xplot, qlo, qhi, color=color, alpha=0.5, label=f'A{loiqr}')

    #uncertainty metric
    area_iqr = find_area_iqr(xplot, yplot, lo=lo, hi=hi)
    if include_area_iqr:
        ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % np.round(area_iqr,2)}", ha='right', va='top', transform=ax.transAxes)

    # format
    ax.get_yaxis().set_visible(False)
    ax.get_xaxis().set_visible(True)
    ax.legend(loc='upper left', bbox_to_anchor=(1,0.75));
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.set_ylim(0,)
    # ax.vlines(x=0, ymin=ax.get_ylim()[0], ymax=ax.get_ylim()[1], color='black');
    # ax.hlines(y=0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], color='black');

## def weighted_quantile(X, W, x, output='perc2val'):

In [ ]:
def weighted_quantile(X, W, x, output='perc2val'):
    """
    Given a set of values with weights, returns a value from a percentage, or a percentage from a value
    INPUTS:
    X       values
    W       weights
    x       desired percentage or value
    output  'perc2val' or 'val2perc' (as string)
    
    OUTPUT:
    value or percentage, depending on 'output' variable
    """
    cdf = W.copy()    
    y_cdf, cdf = zip(*sorted(zip(np.append(X, [0]), np.append(cdf, [0])), key=lambda x: x[0]))
    y_cdf, cdf = np.array(y_cdf), np.array(cdf)
    for i in range(len(cdf)-1):
        cdf[i+1] = cdf[i] + W[i]
    cdf[0] = 0
    cdf[-1] = 1
    y_cdf[0] = min(X)
    
    if output == 'perc2val':
        perc2val = scipy.interpolate.interp1d(cdf, y_cdf, assume_sorted=False)
        return perc2val(x)
    
    elif output == 'val2perc':
        val2perc = scipy.interpolate.interp1d(y_cdf, cdf, assume_sorted=False)
        return val2perc(x)
    
    else:
        raise ValueError("output must be 'perc2val' or 'val2perc'")

## def weighted_std(X, W, BW=None):

In [ ]:
def weighted_std(X, W, BW=None):
    """
    Computes the standard deviation of a weighted dataset, optionally adding kernel variance.

    Parameters
    ----------
    X : array_like
        Data values.
    W : array_like
        Weights (same length as X).
    BW : array_like or None
        Optional bandwidths per data point (standard deviations of kernels).
        If None, no kernel variance is added.

    Returns
    -------
    std : float
        Weighted standard deviation (including kernel contribution if BW is provided).
    """
    X = np.array(X).flatten()
    W = np.array(W).flatten()
    W = W/np.sum(W)
    if BW is None:
        BW = np.zeros_like(W)
    else:
        BW = np.array(BW).flatten()
    
    if not(len(X) == len(W) == len(BW)):
        raise ValueError(f"X, W, and BW must have the same length. {len(X)}, {len(W)}, {len(BW)}")

    mean = np.sum(X * W)
    var_data = np.sum(W * (X - mean)**2)
    var_kernel = np.sum(W * BW**2)
    std = np.sqrt(var_data + var_kernel)
    return std

## def weighted_bw(X, W, bw_method='silverman'):

In [ ]:
def weighted_bw(X, W, bw_method='silverman'):
    """
    Calculates bandwidth using Silverman's or Scott's method, adjusted for weighted data.

    Parameters
    ----------
    X : array_like
        Data values.
    W : array_like
        Weights for each data point.
    bw_method : str
        'silverman' or 'scott'

    Returns
    -------
    bw : float
        Bandwidth estimate (scalar)
    """
    X = np.array(X).flatten()
    W = np.array(W).flatten()
    W = W/sum(W)
        
    if len(X) != len(W):
        raise ValueError(f'Length of X and W must be equal: {len(X)}, {len(W)}')
    
    std = weighted_std(X, W)
    iqr = weighted_quantile(X, W, 0.75, output='perc2val') - weighted_quantile(X, W, 0.25, output='perc2val')
    if iqr == 0 or not np.isfinite(iqr):
        iqr = std * 1.34

    #effective sample size (n_eff) used since len(X) may overstate the sample size with uneven weights
    n_eff = (np.sum(W))**2 / np.sum(W**2)
    # n_eff = len(X)
    
    if bw_method=='silverman':
        bw = 0.9 * np.min([std, iqr/1.34]) * n_eff**-0.2
    elif bw_method=='scott':
        bw = 1.06 * std * n_eff**-0.2
    else:
        raise ValueError(f'bw_method must be silverman or scott instead of {bw_method}')
    
    return bw

## def bw_dirichlet(X, Wavg, BW_factors, Wrun, bw_method='silverman'):

In [ ]:
def bw_dirichlet(X, Wavg, BW_factors, Wrun, bw_method='silverman'):
    """
    Estimates bandwidths for a weighted KDE with variable bandwidths per point.
    Uses a baseline estimate (weighted_bw), and adjusts only if necessary to meet target KDE std.

    Parameters
    ----------
    X : array_like
        Data points.
    Wavg : array_like
        Vector of average weights.
    BW_factors : array_like
        Relative bandwidth multipliers (mean = 1).
    Wrun : array_like
        Sampled weights from Dirichlet distribution.
    bw_method : str
        Method for baseline bandwidth estimate ('silverman' or 'scott').

    Returns
    -------
    BW : ndarray
        Bandwidths to use for KDE (standard deviations of each kernel).
    """
    # Flatten and normalize
    X = np.array(X).flatten()
    Wavg = np.array(Wavg).flatten()
    Wrun = np.array(Wrun).flatten()
    BW_factors = np.array(BW_factors).flatten()

    Wavg = Wavg / np.sum(Wavg)
    Wrun = Wrun / np.sum(Wrun)
    BW_factors = BW_factors / np.mean(BW_factors)

    # Step 1: Compute target variance using reference weights (Wavg)
    bw_base = weighted_bw(X, Wavg, bw_method)  # based on reference weights
    std_target = weighted_std(X, Wavg, BW_factors * bw_base)
    var_target = std_target**2

    # Step 2: Try using weighted_bw with Wrun
    bw = weighted_bw(X, Wrun, bw_method)
    std_data = weighted_std(X, Wrun)
    var_data = std_data**2

    std_kernel = bw * BW_factors
    var_kernel = np.sum(Wrun * std_kernel**2)

    var_kde_guess = var_data + var_kernel
    
    #verify weighted_std and manual calculation are getting the same answer
    a = np.round(var_kde_guess, 4)
    b = np.round(weighted_std(X, Wrun, bw * BW_factors)**2, 4)
    if  a != b :
        raise ValueError(f'Should be 1: {a} / {b}')

    # Step 3: Accept guess if it meets or exceeds target
    if var_kde_guess >= var_target:
        # print(f'bw_base / bw: {bw_base} / {bw}')
        # print(f'bw_dirichlet var_data: {var_data}')
        # print(f'bw_dirichlet var_kernel: {var_kernel}')
        # print(f'bw_dirichlet var_target: {var_target}')
        # print('')
        return bw * BW_factors  # Accept the guess
    
    # Step 4: Otherwise, solve for exact bw to hit target    
    else:
        bw = np.sqrt((var_target - var_data) / np.sum(Wrun * BW_factors**2))
        std_kernel = bw * BW_factors
        var_kernel = np.sum(Wrun * std_kernel**2)
        # print(f'bw_base / bw: {bw_base} / {bw}')
        # print(f'bw_dirichlet (else) var_data: {var_data}')
        # print(f'bw_dirichlet (else) var_kernel: {var_kernel}')
        # print(f'bw_dirichlet (else) var_target: {var_target}')
        # print('')
        if np.round(var_kernel + var_data, 4) < np.round(var_target, 4):
            raise ValueError(f'bw_dirichlet function didnt work. var_kernel+var_data={var_kernel+var_data}; var_target={var_target}')
        return bw * BW_factors

    raise ValueError(f'Failed to return a value')


    # log and verify results
    # if np.round(var_kernel + var_data,4) < np.round(var_target,4):
    #     print(f'ELSE')
    #     print(f'{var_target} = var_target')
    #     print(f'{var_data+var_kernel} = var_data + var_kernel = {var_data} + {var_kernel}')
        # print('')

## def heterogeneous_kde_std(x, w, h):

In [ ]:
# check if algebraic actually equals numerical
def heterogeneous_kde_std(x, w, h):
    w = np.array(w)
    w = w / np.sum(w)
    mu = np.sum(w * x)
    data_var = np.sum(w * (x - mu)**2)
    kernel_var = np.sum(w * h**2)
    return np.sqrt(data_var + kernel_var)

# Xtest = np.array([1.0, 2.0, 3.0])
# Wtest = np.array([0.2, 0.5, 0.3])
# BWtest = np.array([0.1, 0.4, 0.2])
# std_kde = heterogeneous_kde_std(Xtest, Wtest, BWtest)
# print(std_kde)

# xtest = np.linspace(0,5, 1_001)
# ytest = np.zeros_like(xtest)
# for x, w, bw in zip(Xtest, Wtest, BWtest):
#     ytest += norm.pdf(xtest, x, bw)*w

# fig, ax = plt.subplots()
# ax.plot(xtest, ytest)

# print(weighted_std(xtest, ytest))


## def effective_variance(x, pdf):

In [ ]:
def effective_variance(x, pdf):
    norm = np.trapz(pdf, x)
    pdf_norm = pdf / norm
    mu = np.trapz(x * pdf_norm, x)
    return np.trapz(pdf_norm * (x - mu)**2, x)

## def find_area_iqr(xplot, yplot, lo=0.25, hi=0.75):

In [ ]:
def find_area_iqr(xplot, yplot, lo=0.25, hi=0.75):
    """
    INPUT
        xplot    1-d array with x values
        yplot    2-d array with multiple sets of y-values. returned from kl2()
        lo       lower quantile
        hi       upper quantile
    
    OUTPUT
        area_iqr   array between upper and lower quantiles for the plot. Higher metric means greater uncertainty about the probabilistic model
    """
    qlo = np.quantile(yplot, lo, axis=1)
    qhi = np.quantile(yplot, hi, axis=1)
    return np.trapz(qhi, xplot) - np.trapz(qlo, xplot)

# SANDBOX

## Testing different methods of BW. var_dirichlet > var_original, always bw_base, recalculate bw with each new W

Takeaway - best to use a baseline bandwidth. There's reasoning to use the Dirichlet method, but it's needlessly complicated

In [ ]:
%%time
# var_dirichlet > var_original
xplot = np.linspace(0,30,1_001)
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
alpha = np.ones_like(X)
BW_factors = np.ones_like(X)

alpha = np.random.uniform(1,5,len(X))
alpha = alpha / np.min(alpha)

BW_factors = np.random.uniform(0.9,1.1,len(X))
BW_factors = BW_factors / np.mean(BW_factors)

print('alpha')
print(alpha)
print('')
print('BW_factors')
print(BW_factors)
bw_base = weighted_bw(X, alpha / np.sum(alpha), bw_method='silverman')
ytotals = [[],[],[]]

# run all three analyses
for _ in range(250):
    W = np.random.dirichlet(np.ones(len(X)))
    bw_basenew = weighted_bw(X, W, bw_method='silverman')
    
    BW_dir = bw_dirichlet(X, alpha, BW_factors, Wrun=W, bw_method='silverman')
    BW_base = BW_factors / np.mean(BW_factors) * bw_base
    BW_dyn = BW_factors / np.mean(BW_factors) * bw_basenew
    
    for i, BW in enumerate([BW_dir, BW_base, BW_dyn]):
        # create each PDF and ensure it's valid and sums to 1
        yi = norm.pdf(xplot.reshape(-1,1), X, BW)*W
        # yplot[run] = sum(yi.T)
        yplot = yi.sum(axis=1)
        yplot /= np.trapz(yplot, xplot)

        ytotals[i].append(yplot)

# plot all three versions
titles = ['dirichlet', 'base BW', 'dynamic base BW']
fig, axes = plt.subplots(len(ytotals), 1, figsize=(6,6), gridspec_kw=dict(hspace=0.5), sharex=True)
for i, ytotal in enumerate(ytotals):
    ax = axes[i]
    ytotal = np.array(ytotal).T
    ax.plot(xplot, ytotal, linewidth=0.1, color='black');
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_title(titles[i])
    ax.set_ylim(0,)


## Getting all EPDs by year

In [ ]:
# %%time
# #look at all files and only pull the most recent for each type of analysis 
# import os
# filepath = '../EPDsFromEC3/EPD_Output'
# files = [f'{filepath}/{ele}' for ele in os.listdir(filepath) if ele.startswith('EPDs_')]
# # files.sort()
# # dft = pd.DataFrame([ele[:-5].split('_')[-2:] for ele in files], columns=['material', 'date'])
# # dft['date'] = pd.to_datetime(dft['date'])
# # dft['file'] = files
# # dft = dft.sort_values('date', ascending=False)
# # lst = list(dft['material'].unique())
# # lst.sort()
# # lst

# dates = []
# uuids = []
# files.sort()
# files = [ele for ele in files if ele.find('_concrete_') == -1]
# feat1 = 'date_of_issue'
# feat2 = 'open_xpd_uuid'
# for file in files:
#     print(file, ' - ', len(dates),'/',len(uuids))
#     dft = pd.read_excel(file)
#     if feat1 in dft.columns and feat2 in dft.columns:
#         dft = dft[[feat1, feat2]]
#         dates += list(dft.dropna()['date_of_issue'].values)
#         uuids += list(dft.dropna()['open_xpd_uuid'].values)  

# dft = pd.DataFrame({'date':dates, 'uuid':uuids})
# dft = dft.drop_duplicates(subset=['uuid'])
# dft['year'] = dft['date'].str[:4]
# dft['year'].value_counts().sort_index()

# # year
# # 2011        1
# # 2012        2
# # 2013       10
# # 2014        7
# # 2015       74
# # 2016       54
# # 2017       48
# # 2018       84
# # 2019     4204
# # 2020     3241
# # 2021    24694
# # 2022    16713
# # 2023    16321
# # 2024     1485

## Visualizing stick-breaking

In [ ]:
# Visualize stick-breaking
alphas = (1,1,1)
results = []
for _ in range(50):
    results.append(np.random.dirichlet(alphas))
results = np.array(results)

fig, ax = plt.subplots()

pd.DataFrame(results).plot(kind='barh', stacked=True, ax=ax, width=0.8)
ax.get_legend().set_visible(False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(0,1)

In [ ]:
# Visualize stick-breaking
alphas = np.ones(10)
results = []
for _ in range(50):
    results.append(np.random.dirichlet(alphas))
results = np.array(results)

fig, ax = plt.subplots()

pd.DataFrame(results).plot(kind='barh', stacked=True, ax=ax, width=0.8)
ax.get_legend().set_visible(False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(0,1)

##  Demonstrating Dirichlet method

In [ ]:
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
bw = np.std(X)*len(X)**-0.2
xplot = np.linspace(1,30,1001)

fig, ax = plt.subplots()
for _ in range(1000):
    yi = np.zeros_like(xplot)
    # W = 0.05 + np.random.dirichlet(np.ones_like(X))*0.55
    W = np.random.dirichlet(np.ones_like(X))
    yi = norm.pdf(xplot.reshape(-1,1), X, bw)*W
    ytotal = yi.sum(axis=1)
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linewidth=0.025, color='black')

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([])
ax.set_ylim(0,)

In [ ]:
alpha = np.array([5,2,3,6,1,4])
# X = np.array([1,2,3,5,6,7])
X = np.arange(len(alpha))
Xstrip = np.array([[ele]*results.shape[0] for ele in X]).flatten()
yerrs = 0.5*alpha**0.5
target = 1.25
alpha_multipliers = [1, 2, 4]

fig, ax = plt.subplots(1,1, sharex=False, gridspec_kw=dict(hspace=0.7), figsize=(6,3))

#################### FIRST PLOT ####################################
ymax = 0.75
labels = [lo1, lo2, lo3]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for offset, am in enumerate(alpha_multipliers):
        if am < 1:
            alpha = alpha**am
        else:
            alpha = alpha*am
        results = []
        for _ in range(10_000):
            wt = np.random.dirichlet(alpha)
            results.append(wt)
        results = np.array(results)
        Xstrip = np.array([[ele]*results.shape[0] for ele in X]).flatten()
        Xstrip = Xstrip*(len(alpha_multipliers)+2) + offset
        sns.stripplot(x=Xstrip, y=results.T.flatten(), ax=ax, palette='tab10', native_scale=True, s=0.25, jitter=0.05)
        
        for x, a, color in zip(set(Xstrip), alpha, sns.color_palette('tab10', len(alpha))):
            ax.text(x=x, y=ymax, s="{:.1f}".format(np.round(a,1)), color=color, ha='center', va='bottom', fontsize=10)
            # if offset==0 and x==list(set(Xstrip))[0]:
            if x==list(set(Xstrip))[0]:
                ax.text(x=x-1, y=ymax, s=f'α{labels[offset]}{locomma}{loi}=', color=color, ha='right', va='bottom', fontsize=10)
        ymax -= 0.08


ax.set_xticks(X*(len(alpha_multipliers)+2)+1, X)
ax.set_ylim(0,)
ymin, ymax = ax.get_ylim()
ax.text(x=0, y=1.2, s=f'(a) Sampled weights based on three sets of alpha', ha='left', va='bottom', transform=ax.transAxes, fontsize=12)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)

## Exploring gamma distribution

In [ ]:
#looking at how the gamma distribution ch anges at a=1, a<1, a>1
A = [0.9,1,1.1]
n = 10_000
xplot = np.linspace(1/n,5,n)

fig, ax = plt.subplots()
for a in A:
    yplot = scipy.stats.gamma.pdf(x=xplot, a=a)
    # yplot /= max(yplot)
    ax.plot(xplot,yplot, label=a)
ax.legend()
ax.set_ylim(0,2)

In [ ]:
# looking at how the gamma distribution changes at different shapes and scales
factor = 100
fig, ax = plt.subplots(figsize=(6,2))
plt.hist(np.random.gamma(shape=100, scale=1, size=100_000), bins=100, density=True, alpha=0.5, label='Regular');
plt.hist(np.random.gamma(shape=100*factor, scale=1/factor, size=100_000), bins=100, density=True, alpha=0.5, label='With Factor');
plt.legend()

# METHODS

## Showcasing the Dirichlet distribution with different alphas

In [ ]:
# Showcasing Dirichlet distribution with different alphas - triangular plot with 2/3 weights shown

alphas = [(1,1,1), (0.25,0.25,0.25), (4,4,4), (4,1,1), (1,0.25,0.25), (16,4,4)]
# alphas = [(0.2,0.05,0.05), (1,0.25,0.25), (4,1,1), [8,2,2]]
# alphas = [(5,0.5,0.5), (7.5,0.75,0.75), (10,1,1), [12,1.2,1.2]]
labels = [r"$\bf{ (a) }$",
          r"$\bf{ (b) }$",
          r"$\bf{ (c) }$",
          r"$\bf{ (d) }$",
          r"$\bf{ (e) }$",
          r"$\bf{ (f) }$",
         ]

nrows = 2
ncols = int(len(alphas)/nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=(6,3), sharex=False, sharey=False, gridspec_kw=dict(hspace=0.5, wspace=0.2))

r = 0
c = 0

for alpha, label in zip(alphas, labels):
    ax = axes[r,c]
    xs = []
    ys = []
    zs = []
    for _ in range(10_000):
        x, y, z = np.random.dirichlet(alpha)
        xs.append(x)
        ys.append(y)
    ax.scatter(xs, ys, s=0.01, color='tab:blue')
    if r==0 and c==0: alpha = f'α={alpha}'
    ax.set_title(alpha, fontsize=10, loc='left')
    if r==nrows-1 and c==0:
        ax.set_xticks([0,1], [0,1])
        ax.set_yticks([0,1], [0,1])
        ax.set_xlabel(f'W{lo1}')
        ax.set_ylabel(f'W{lo2}', rotation=0)
    else:
        ax.set_xticks([0,1], [])
        ax.set_yticks([0,1], [])
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    ax.spines[['top', 'right', 'left','bottom']].set_visible(False)
    ax.text(x=-0.20, y=0.95, s=label, ha='right', va='center', transform=ax.transAxes, fontsize=10)

    if c==ncols-1:
        r += 1
        c = 0
    else:
        c += 1

In [ ]:
# Showcasing Dirichlet distribution with different alphas - barycentric/ternary plot with 3/3 weights shown
nrows = 2
ncols = 3
nplot = 0


fig = plt.figure(figsize=(6,3))
fig.subplots_adjust(left=0.075, right=0.85, wspace=0.3)

# nplot += 1
# t0, l0, r0 = np.random.dirichlet(alpha=(1.0, 1.0, 1.0), size=10_000).T
# ax = fig.add_subplot(nrows, ncols, nplot, projection="ternary")
# ax.scatter(t0, l0, r0, s=1)

alphas = [
    (1,1,1),
    (0.25, 0.25, 0.25),
    (4, 4, 4),
    (4,1,1),
    (1, 0.25, 0.25),
    (16, 4, 4),
]

labels = [r"$\bf{ (a) }$",
          r"$\bf{ (b) }$",
          r"$\bf{ (c) }$",
          r"$\bf{ (d) }$",
          r"$\bf{ (e) }$",
          r"$\bf{ (f) }$",
         ]

for alpha, label in zip(alphas, labels):
    nplot += 1
    #plot
    t0, l0, r0 = np.random.dirichlet(alpha=alpha, size=10_000).T
    ax = fig.add_subplot(nrows, ncols, nplot, projection="ternary")
    ax.scatter(t0, l0, r0, s=0.005, alpha=1, marker='.')

    #format
    ticks = [0.0, 0.25, 0.5, 0.75, 1.0]
    lw = 0.25
    if alpha == alphas[0]:
        labels = ['0.0', '0.25', '0.50', '0.75', '1.0']
        fontsize = 6
        ax.set_tlabel("$W_1$", loc='right')
        ax.set_llabel("$W_2$", loc='right')
        ax.set_rlabel("$W_3$", loc='right')
        alpha = f'α={alpha}'
        ax.taxis.set_tick_params(width=lw)
        ax.laxis.set_tick_params(width=lw)
        ax.raxis.set_tick_params(width=lw)        
        
    else:
        labels = ['', '', '', '', '']
        ax.taxis.set_tick_params(width=0.)
        ax.laxis.set_tick_params(width=0.)
        ax.raxis.set_tick_params(width=0.)
        
    ax.taxis.set_ticks(ticks, labels=labels, fontsize=fontsize)
    ax.laxis.set_ticks(ticks, labels=labels, fontsize=fontsize)
    ax.raxis.set_ticks(ticks, labels=labels, fontsize=fontsize)
    
    position = 'corner'
    # position = 'tick1'
    ax.taxis.set_label_position(position)
    ax.laxis.set_label_position(position)
    ax.raxis.set_label_position(position)
    
#     mode = 'axis'
#     ax.taxis.set_label_rotation_mode(mode)
#     ax.laxis.set_label_rotation_mode(mode)
#     ax.raxis.set_label_rotation_mode(mode)

    ax.grid(color='black', zorder=3, linewidth=lw)
    ax.set_title(alpha, fontsize=10, loc='left')
    ax.text(x=-0.20, y=0.95, s=label, ha='right', va='center', transform=ax.transAxes, fontsize=10)


plt.subplots_adjust(hspace=0.75, wspace=0.2)
plt.show()

## Visualizing different bandwidths and widths

In [ ]:
# show equal and variable bandwidths and weights
X = [1, 4, 15]
Ws = [[1,1,1],[17,33,50]]
BW_factors_sets = [[1,1,1],[0.75, 1, 1.25]]
xplot = np.linspace(-20, 35, 1000)


nrows = 2
ncols = 2
fig, axes = plt.subplots(nrows,ncols, sharex=True)

for (r, W), (c, BW_factors) in product(enumerate(Ws), enumerate(BW_factors_sets)):
    ax = axes[r,c]
    W /= sum(np.array(W))
    bw_base = weighted_bw(X, W, bw_method='scott')
    BW = bw_base*np.array(BW_factors)

    ysum = np.zeros_like(xplot)
    for x, w, bw in zip(X, W, BW):
        yplot = norm.pdf(xplot, x, bw)*w
        ysum += yplot
        if x == X[0]:
            label = 'Kernels'
        else:
            label = ''
        ax.plot(xplot, yplot, color='tab:blue', linestyle='--', linewidth=1, label=label)

    ax.plot(xplot, ysum, color='black', label='Resulting PDF')
    ymin, ymax = ax.get_ylim()
    ax.scatter(X, [0.05*ymax]*len(X), marker='|', s=250)
    
    #format
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.set_ylim(0,)


fontsize = 12
ax = axes[0,0]
ax.set_title('Equal Kernel Bandwidths', fontsize=fontsize)
ax.text(x=0, y=0.5, s='Equal Kernel Weights', ha='right', va='center', transform=ax.transAxes, fontsize=fontsize)

ax = axes[0,1]
ax.set_title(f'Variable Kernel Bandwidths\n {", ".join([f"{ele:.2f}" for ele in BW_factors_sets[1]])}', fontsize=fontsize)
ax.legend(bbox_to_anchor=(1,0.5), loc='center left')

ax = axes[1,0]
ax.text(x=0, y=0.5, s=f'Variable Kernel Weights', ha='right', va='center', transform=ax.transAxes, fontsize=fontsize);
ax.text(x=0, y=0.35, s=', '.join([f"{int(np.round(ele,2)*100)}%" for ele in Ws[1]/sum(np.array(Ws[1]))]), ha='right', va='center', transform=ax.transAxes, fontsize=fontsize);

labels = [r"$\bf{ (a) }$",
          r"$\bf{ (b) }$",
          r"$\bf{ (c) }$",
          r"$\bf{ (d) }$",
         ]

r, c = 0, 0
for label in labels:
    ax = axes[r,c]
    ax.text(x=0.15, y=0.85, s=label, fontsize=fontsize, transform=ax.transAxes)
    if c == ncols-1:
        c = 0
        r += 1
    else:
        c += 1

plt.savefig('outputs/figures/VisualizeEqualVariableBandwidthsWeights.jpg', bbox_inches='tight')


## SCENARIO 1: Everything is known

In [ ]:
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
       0.89708815, 0.91940053, 1.04995333, 1.12918731])
BW_factors = BW_factors / np.mean(BW_factors)
W = np.array([8, 17, 12, 20,  7, 13,  9,  9,  5])
W = W/sum(W)
bw_base = weighted_bw(X, W, bw_method='silverman')
xplot = np.linspace(min(X)-bw_base*3, max(X)+bw_base*3, 1_001)
colors = sns.color_palette('tab10', len(X))

fig, ax = plt.subplots(figsize=(6,4))
ytitle = 1.2


ytotal = np.zeros_like(xplot)
for x, w, bw_factor, color in zip(X, W, BW_factors, colors):
    yplot = norm.pdf(xplot, x, bw_factor*bw_base)*w
    ytotal += yplot
    ax.plot(xplot, yplot, linestyle='--', linewidth=0.5, color=color)
    ax.scatter(x, -0.0035, marker='|', color=color, s=50)  
    ax.text(x, -0.007, f'{int(np.round(w*100,0))}%', ha='center', va='top', color=color, fontsize=6)
ytotal /= np.trapz(ytotal, xplot)
ax.plot(xplot, ytotal, linestyle='-', linewidth=1, color='black', label='Resulting PDF')

#format
xmin, xmax = ax.get_xlim()
ax.hlines(0, xmin, xmax, color='black', linewidth=1)
ax.hlines(0, xmin, xmax, color='gray', linestyle='--', linewidth=0.5, zorder=0, label='Individual Kernels (ECCs)')
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_xlabel(f'Embodied Carbon Coefficient (kgCO{lo2}e/unit)')
ax.legend(bbox_to_anchor=(-0.02,1), loc='upper left')
ax.text(x=0, y=1.0, s='Scenario 1', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)

xplot_s1 = xplot
yplot_s1 = ytotal

plt.savefig('outputs/figures/SCENARIO1_EverythingIsKnown.jpg', bbox_inches='tight')


## SCENARIO 2: Kernels and product level uncertainty
- alpha / weight
- bandwidth / product-level uncertainty

In [ ]:
# fig, ax = plt.subplots()


# ax.plot(xplot, Y.T, color='black', linewidth=0.025);

# qlo = np.quantile(Y, 0.25, axis=0)
# qhi = np.quantile(Y, 0.75, axis=0)
# ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=0.5)

# print('A_IQR')
# print(np.trapz(qhi, xplot) - np.trapz(qlo, xplot))
# print('')

In [ ]:
%%time


X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
       0.89708815, 0.91940053, 1.04995333, 1.12918731])
BW_factors = BW_factors / np.mean(BW_factors)
alpha = np.ones_like(X)
W = alpha/np.sum(alpha)
bw_base = weighted_bw(X, W, bw_method='silverman')
BW = BW_factors*bw_base
std_base = weighted_std(X, W)
xplot = np.linspace(min(X)-std_base*3, max(X)+std_base*3, 1_001)
colors = sns.color_palette('tab10', len(X))

nrows = 2
ncols = 1
fig, axes = plt.subplots(nrows, ncols, figsize=(6,4),
                         gridspec_kw=dict(hspace=0.75, wspace=0.1))
ytitle = 1.2
nruns = 1_000
lw = 0.01

alpha_aiqr = 1.0

####################### TOP PLOT ######################################################
ax = axes[0]

Ws = []
Y = []
# run analysis
for _ in range(nruns):
    Wrun = np.random.dirichlet(alpha)
    Ws.append(Wrun)
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
        if _==0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
    Y.append(ytotal)

# format arrays and save results
Y = np.array(Y)
Ws = np.array(Ws)
yplot_s21 = Y

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
aiqr = np.trapz(qhi, xplot) - np.trapz(qlo, xplot)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f"A{loiqr}")
ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % aiqr}", ha='right', va='top', fontsize=8, transform=ax.transAxes)
print('A_IQR')
print(aiqr)
print('')


# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean PDF')

#format
ax.text(x=0, y=ytitle, s=r"$\bf{ (a) }$" + f' Kernel weights uniformly sampled with Dirichlet distribution', ha='left', va='top', fontsize=10, transform=ax.transAxes)
ax.legend(loc='upper left', bbox_to_anchor=(0,1), fontsize=8)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_xlabel(f'Embodied Carbon Coefficient (kgCO{lo2}e/unit)')



####################### BOTTOM PLOT ######################################################
ax = axes[1]

#plot
sns.stripplot(Ws, ax=ax, size=0.5)
ax.scatter(np.arange(len(alpha)), alpha/sum(alpha), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, y, s, c in zip(np.arange(len(alpha)), alpha/sum(alpha), alpha, colors):
    ax.text(x=x-0.2, y=y, s=f'{np.round(s/sum(alpha)*100,1)}%', color=c, ha='right', va='center', fontsize=6)

#add label legend
y = 0.6
ax.text(x=0.2, y=y, s='Mean Weight', ha='right', va='center', fontsize=8, transform=ax.transAxes, color='black')
ax.scatter(x=0.225, y=y, marker='_', s=200, color='black',transform=ax.transAxes)

#add BW labels

#format
# ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (b) }$" + ' Sampled kernel weights', ha='left', va='top', fontsize=10, transform=ax.transAxes)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(0,);
ybmin, ybmax = ax.get_ylim()


ax = axes[0]
ax.text(x=0, y=ytitle+0.15, s='Scenario 2', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)

plt.savefig('outputs/figures/SCENARIO2_KernelsAndProductLevelUncertainty.jpg', bbox_inches='tight')

## SCENARIO 3: We know all the quantities and product level uncertainty
- production quantities
- grouped production quantities

In [ ]:
%%time

X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
alpha = np.ones_like(X)
W = alpha / np.sum(alpha)
BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
       0.89708815, 0.91940053, 1.04995333, 1.12918731])
BW_factors = BW_factors / np.mean(BW_factors)
bw_base = weighted_bw(X, W, bw_method='silverman')
BW = BW_factors*bw_base
std_base = weighted_std(X, W)
xplot = np.linspace(min(X)-std_base*3, max(X)+std_base*3, 1_001)
colors = sns.color_palette('tab10', len(X))

nruns = 1_000
lw = 0.01

alpha_iqr = 1.0


nrows = 2
ncols = 2
fig, axes = plt.subplots(nrows,ncols,figsize=(12,4), gridspec_kw=dict(hspace=0.75, wspace=0.1))
ytitle = 1.2
######################## TOP LEFT PLOT ########################
ax = axes[0,0]
gc1 = 0.5
gc2 = 0.5

#plot
Ws = []
Y = []
for _ in range(nruns):
    Wrun = np.concatenate([np.random.dirichlet(np.ones(3))*gc1, np.random.dirichlet(np.ones(6))*gc2])
    Ws.append(Wrun)
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, np.ones_like(X), BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
        if _==0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)       
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
    Y.append(ytotal)
# format arrays
Y = np.array(Y)
Ws = np.array(Ws)

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
aiqr = np.trapz(qhi, xplot) - np.trapz(qlo, xplot)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f"A{loiqr}")
ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % aiqr}", ha='right', va='top', fontsize=8, transform=ax.transAxes)
print('A_IQR')
print(aiqr)
print('')

# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean PDF')

#added text
ax.text(x=X[2], y=-0.025, s=f'Group 1 = {int(gc1*100)}%', ha='right', va='top')
ax.text(x=X[3], y=-0.025, s=f'Group 2 = {int(gc2*100)}%', ha='left', va='top')

#format
ax.text(x=0, y=ytitle, s=r"$\bf{ (a) }$" + ' Grouped kernel weights are known', ha='left', va='top', fontsize=10, transform=ax.transAxes)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);

ax.legend(loc='upper left', bbox_to_anchor=(0,0.9), fontsize=8)



######################## BOTTOM LEFT ########################
ax = axes[1,0]

#plot
sns.stripplot(Ws, ax=ax, size=0.5)
ax.scatter(np.arange(len(X)), np.mean(Ws, axis=0), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, w, c in zip(np.arange(len(X)), np.mean(Ws, axis=0), colors):
    if w > 0.1:
        w = gc1/3
    else:
        w = gc2/6
    ax.text(x=x-0.2, y=w, s=f'{np.round(w*100,1)}%', color=c, ha='right', va='center', fontsize=6)

#add label legend
y = 0.7
ax.text(x=0.2, y=y, s='Mean Weight', ha='right', va='center', fontsize=8, transform=ax.transAxes, color='black')
ax.scatter(x=0.225, y=y, marker='_', s=200, color='black',transform=ax.transAxes)

#format
# ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (b) }$" + ' Sampled kernel weights of (a)', ha='left', va='top', fontsize=10, transform=ax.transAxes)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(0,);




######################## TOP RIGHT PLOT ########################
ax = axes[0,1]
gc1 = 0.5
gc2 = 0.5

#plot
Ws = []
Y = []
for _ in range(nruns):
    # Wrun = np.concatenate([np.ones(3)/3*gc1, np.random.dirichlet(np.ones(6))*gc2])
    Wrun = np.concatenate([np.random.dirichlet(np.ones(3))*gc1, np.ones(6)/6*gc2])
    Ws.append(Wrun)
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, np.ones_like(X), BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
        if _==0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)       
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
    Y.append(ytotal)
# format arrays
Y = np.array(Y)
Ws = np.array(Ws)

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
aiqr = np.trapz(qhi, xplot) - np.trapz(qlo, xplot)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f"A{loiqr}")
ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % aiqr}", ha='right', va='top', fontsize=8, transform=ax.transAxes)
print('A_IQR')
print(aiqr)
print('')

# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean PDF')

#added text
ax.text(x=X[2], y=-0.025, s=f'Group 1 = {int(gc1*100)}%', ha='right', va='top')
ax.text(x=X[3], y=-0.025, s=f'Exact Weights Known', ha='left', va='top')

#format
ax.text(x=0, y=ytitle, s=r"$\bf{ (c) }$" + ' Exact and grouped kernel weights are known', ha='left', va='top', fontsize=10, transform=ax.transAxes)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);


######################## BOTTOM RIGHT ########################
ax = axes[1,1]

#plot
sns.stripplot(Ws, ax=ax, size=0.5)
ax.scatter(np.arange(len(X)), np.mean(Ws, axis=0), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, w, c in zip(np.arange(len(X)), np.mean(Ws, axis=0), colors):
    if w > 0.1:
        w = gc1/3
    else:
        w = gc2/6
    ax.text(x=x-0.2, y=w, s=f'{np.round(w*100,1)}%', color=c, ha='right', va='center', fontsize=6)


#add label legend
# ax.text(x=0.2, y=0.7, s='α', ha='right', va='bottom', fontsize=8, transform=ax.transAxes, color='black')
# ax.text(x=0.2, y=0.7, s='Mean Weight', ha='right', va='top', fontsize=8, transform=ax.transAxes, color='black')
# ax.scatter(x=0.225, y=0.7, marker='_', s=200, color='black',transform=ax.transAxes)

#format
# ax.set_title('(d) Sampled kernel weights of (c)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (d) }$" + ' Sampled kernel weights of (c)', ha='left', va='top', fontsize=10, transform=ax.transAxes)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(0,);


# set bottom row ylims to be equal
ymax = np.max([axes[1,0].get_ylim()[1], axes[1,1].get_ylim()[1]])
axes[1,0].set_ylim(0, ymax)
axes[1,1].set_ylim(0, ymax)

ax = axes[0,0]
ax.text(x=0, y=ytitle+0.15, s='Scenario 3.1', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)
ax = axes[0,1]
ax.text(x=0, y=ytitle+0.15, s='Scenario 3.2', ha='left', va='bottom', fontsize=12, transform=ax.transAxes);


plt.savefig('outputs/figures/SCENARIO3_QuantitiesKnown.jpg', bbox_inches='tight')


In [ ]:
# %%time

# X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
# # alpha = np.arange(1,len(X)+1)
# # alpha = np.array([1]*3 + [4]*6)
# alpha = np.ones_like(X)
# W = alpha / np.sum(alpha)
# BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
#        0.89708815, 0.91940053, 1.04995333, 1.12918731])
# BW_factors = BW_factors / np.mean(BW_factors)
# std_base = weighted_std(X, alpha/sum(alpha))
# xplot = np.linspace(min(X)-std_base*3, max(X)+std_base*3, 1_001)
# colors = sns.color_palette('tab10', len(X))

# nruns = 1_000
# lw = 0.01


# nrows = 2
# ncols = 2
# fig, axes = plt.subplots(nrows,ncols,figsize=(12,4), gridspec_kw=dict(hspace=0.2, wspace=0.1))
# ytitle = 1.2
# ######################## TOP LEFT PLOT ########################
# ax = axes[0,0]

# #plot
# Ws = []
# Y = []
# for _ in range(nruns):
#     Wrun = np.random.dirichlet(alpha)
#     Ws.append(Wrun)
#     ytotal = np.zeros_like(xplot)
#     BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
#     for x, w, bw, color in zip(X, Wrun, BW, colors):
#         yplot = norm.pdf(xplot, x, bw)*w
#         ytotal += yplot
#         ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
#         if _==0:
#             ax.scatter(x, -0.01, marker='|', color=color, s=50)       
#     ytotal /= np.trapz(ytotal, xplot)
#     ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
#     Y.append(ytotal)
# Y = np.array(Y)
# Ws = np.array(Ws)

# ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean Resulting PDF')

# #format
# # ax.set_title('(a) Resulting PDF with kernel weights produced \n with the Dirichlet distribution', loc='left')
# # ax.text(x=0, y=ytitle, s=r"$\bf{ (a) }$" + ' Resulting PDF when kernel weights of \n individual kernels are known', ha='left', va='top', fontsize=10, transform=ax.transAxes)
# ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
# ax.get_yaxis().set_visible(False)
# ax.set_xticks([]);
# # ax.set_xlabel('Kernel Locations')
# ax.legend(loc='upper left', bbox_to_anchor=(0,0.9), fontsize=8)



# ######################## BOTTOM LEFT ########################
# ax = axes[1,0]

# #plot
# sns.stripplot(Ws, ax=ax, size=0.5)
# ax.scatter(np.arange(len(alpha)), alpha/sum(alpha), marker='_', s=200, color=colors)

# #add labels to each strip plot
# for x, y, s, c in zip(np.arange(len(alpha)), alpha/sum(alpha), alpha, colors):
#     # ax.text(x=x-0.2, y=y, s=s, color=c, ha='right', va='bottom', fontsize=6)
#     ax.text(x=x-0.2, y=y, s=f'{np.round(s/sum(alpha)*100,1)}%', color=c, ha='right', va='center', fontsize=6)

# #add label legend
# y = 0.7
# # ax.text(x=0.2, y=y, s='α', ha='right', va='bottom', fontsize=8, transform=ax.transAxes, color='black')
# # ax.text(x=0.2, y=y, s='Mean Weight', ha='right', va='top', fontsize=8, transform=ax.transAxes, color='black')
# # ax.scatter(x=0.225, y=y, marker='_', s=200, color='black',transform=ax.transAxes)

# #format
# # ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
# # ax.text(x=0, y=ytitle, s=r"$\bf{ (b) }$" + ' Sampled kernel weights of (a)', ha='left', va='top', fontsize=10, transform=ax.transAxes)

# ax.spines[['top', 'right', 'left']].set_visible(False)
# ax.get_yaxis().set_visible(False)
# ax.set_xticks([]);
# ax.set_ylim(0,);




# ######################## TOP RIGHT PLOT ########################
# ax = axes[0,1]
# gc1 = 0.6
# gc2 = 0.4

# #plot
# Ws = []
# Y = []
# for _ in range(nruns):
#     Wrun = np.concatenate([np.random.dirichlet(alpha[:3])*gc1, np.random.dirichlet(alpha[3:])*gc2])
#     Ws.append(Wrun)
#     ytotal = np.zeros_like(xplot)
#     BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
#     for x, w, bw, color in zip(X, Wrun, BW, colors):
#         yplot = norm.pdf(xplot, x, bw)*w
#         ytotal += yplot
#         ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
#         if _==0:
#             ax.scatter(x, -0.01, marker='|', color=color, s=50)       
#     ytotal /= np.trapz(ytotal, xplot)
#     ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
#     Y.append(ytotal)
# Y = np.array(Y)
# Ws = np.array(Ws)

# ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean Resulting PDF')

# #added text
# ax.text(x=X[0], y=-0.025, s=f'Group 1 = {int(gc1*100)}%', ha='center', va='top')
# ax.text(x=X[3], y=-0.025, s=f'Group 2 = {int(gc2*100)}%', ha='left', va='top')

# #format
# # ax.text(x=0, y=ytitle, s=r"$\bf{ (c) }$" + ' Resulting PDF when summed kernel weights of \n groups of kernels are known', ha='left', va='top', fontsize=10, transform=ax.transAxes)
# ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
# ax.get_yaxis().set_visible(False)
# ax.set_xticks([]);
# # ax.set_xlabel('Kernel Locations')
# ax.legend(loc='upper right', bbox_to_anchor=(1,0.9), fontsize=8)


# ######################## BOTTOM RIGHT ########################
# ax = axes[1,1]

# #plot
# sns.stripplot(Ws, ax=ax, size=0.5)
# ax.scatter(np.arange(len(alpha)), np.mean(Ws, axis=0), marker='_', s=200, color=colors)

# #add labels to each strip plot
# for x, w, a, c in zip(np.arange(len(alpha)), np.mean(Ws, axis=0), alpha, colors):
#     # ax.text(x=x-0.2, y=w, s=a, color=c, ha='right', va='bottom', fontsize=6)
#     if w > 0.1:
#         w = gc1/3
#     else:
#         w = gc2/6
#     ax.text(x=x-0.2, y=w, s=f'{np.round(w*100,1)}%', color=c, ha='right', va='center', fontsize=6)


# #add label legend
# # ax.text(x=0.2, y=0.7, s='α', ha='right', va='bottom', fontsize=8, transform=ax.transAxes, color='black')
# # ax.text(x=0.2, y=0.7, s='Mean Weight', ha='right', va='top', fontsize=8, transform=ax.transAxes, color='black')
# # ax.scatter(x=0.225, y=0.7, marker='_', s=200, color='black',transform=ax.transAxes)

# #format
# # ax.set_title('(d) Sampled kernel weights of (c)', loc='left')
# # ax.text(x=0, y=ytitle, s=r"$\bf{ (d) }$" + ' Sampled kernel weights of (c)', ha='left', va='top', fontsize=10, transform=ax.transAxes)
# ax.spines[['top', 'right', 'left']].set_visible(False)
# ax.get_yaxis().set_visible(False)
# ax.set_xticks([]);
# ax.set_ylim(0,);


# # ax = axes[0,0]
# # ax.text(x=0, y=ytitle+0.15, s='Scenario 3.1', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)
# # ax = axes[0,1]
# # ax.text(x=0, y=ytitle+0.15, s='Scenario 3.2', ha='left', va='bottom', fontsize=12, transform=ax.transAxes);


# plt.savefig('outputs/figures/SCENARIO3_QuantitiesKnown_uniformalpha.jpg', bbox_inches='tight')


## SCENARIO 4: Trying to get to EVtarget (EVtarget, represented)
- EVtarget (target expected value)
- represented (percentage of weight that is simulated)

In [ ]:
# string = r'$\frac{3}{4}$'
# fig, ax = plt.subplots()

# ax.plot(xplot, yplot)
# ax.text(10, 0.02, string, fontsize=16)
# # ax.text(10, 0.017, f'EV{lobacksim}{lophantom}', fontsize=16)

In [ ]:
%%time
gc1 = 0.5
gc2 = 0.3
rep = gc1 + gc2
sim = 1 - rep

X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
# alpha = np.arange(1,len(X)+1)
# alpha = np.array([1]*3 + [4]*6)
alpha = np.ones_like(X)
BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
       0.89708815, 0.91940053, 1.04995333, 1.12918731])
BW_factors = BW_factors / np.mean(BW_factors)
bw_base = 0.9*np.std(X)*len(X)**-0.2
colors = sns.color_palette('tab10', len(X))
color_evold = sns.color_palette('tab10',len(X)+1)[-1]

nruns = 1_000
xplot = np.linspace(np.min(X-5*BW_factors*bw_base), np.max(X+10*BW_factors*bw_base), 1_001)
alpha_aiqr = 1.0

#plot
EV = []
nrows = 2
ncols = 2
wspace = 0.025
fig, axes = plt.subplots(nrows,ncols, figsize=(12,5), gridspec_kw=dict(hspace=0.75, wspace=wspace))

####################### TOP LEFT PLOT ######################################################
ax = axes[0,0]
Ws = []
Y = []
X1 = []
EV = []
target = 15.5
# run analysis
for _ in range(nruns):
    # plot the known kernels
    Wrun = np.concatenate([np.random.dirichlet(alpha[:3])*gc1, np.random.dirichlet(alpha[3:])*gc2])
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        if _ == 0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)
        ax.plot(xplot, yplot, linestyle='--', linewidth=0.05, color=color)


    # plot the phantom kernel
    ev = sum(X*Wrun)/sum(Wrun)
    EV.append(ev)
    xnew = target + (target-ev)/sim*rep
    X1.append(xnew)
    yplot = norm.pdf(xplot, xnew, bw_base)*sim
    if _ == 0:
        label = f'Phantom Kernel ({int(np.round(sim*100,2))}%)'
        ax.plot(xplot, [0]*len(xplot), linestyle='-', linewidth=0.5, color='purple', zorder=0, label=label)
    else:
        label = ''
        ax.plot(xplot, yplot, linestyle='-', linewidth=0.04, color='purple', zorder=3, label=label, alpha=0.5)

    # plot the resulting PDF
    ytotal += yplot
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=0.01, color='black', alpha=0.5)
    

    Xnew = np.concatenate([X,[xnew]])
    Wnew = np.concatenate([Wrun,[sim]])
    ev = sum(Xnew*Wnew)/sum(Wnew)
    # EV.append(ev)
    Ws.append(Wnew)
    Y.append(ytotal)
# format arrays
Y = np.array(Y)
Ws = np.array(Ws)

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
aiqr = np.trapz(qhi, xplot) - np.trapz(qlo, xplot)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f"A{loiqr}")
ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % aiqr}", ha='right', va='top', fontsize=8, transform=ax.transAxes)
print('A_IQR')
print(aiqr)
print('')


# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean PDF')

#format
ax.hlines(y=0, xmin=np.min(xplot), xmax=np.max(xplot), color='black', linewidth=1.0)
ymin, ymax = ax.get_ylim()
ax.vlines(x=EV, ymin=0, ymax=ymax*0.95, alpha=0.01, color=color_evold)
ax.text(x=np.mean(EV), y=ymax*0.96, s=f'EV{looriginal}', ha='right', va='bottom', fontsize=10, color=color_evold)
ax.vlines(x=target, ymin=0, ymax=ymax*0.95, linewidth=1, color='black')
ax.text(x=target, y=ymax*0.95, s=f'EV{lotarget}', ha='left', va='bottom', fontsize=10, color='black')

ax.text(x=X[2], y=-0.025, s=f'Group 1 = {int(gc1*100)}%', ha='right', va='top', fontsize=8)
ax.text(x=X[3], y=-0.025, s=f'Group 2 = {int(gc2*100)}%', ha='left', va='top', fontsize=8)

ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
# ax.set_ylim(0,);
xleg = 0.75
ax.legend(loc='lower right', bbox_to_anchor=(1, 0.2), fontsize=8);
ax.text(x=0, y=ytitle, s=r"$\bf{ (a) }$" + ' Using phantom kernels to achieve ' + r"$\bf{ slightly }$" f' higher EV{lotarget}', ha='left', va='top', fontsize=10, transform=ax.transAxes)





#################### BOTTOM LEFT PLOT ####################################
ax = axes[1,0]
colors.append('purple')

#plot
sns.stripplot(Ws, ax=ax, size=0.5, palette=colors)
ax.scatter(np.arange(Ws.shape[1]), np.mean(Ws, axis=0), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, w, a, c in zip(np.arange(Ws.shape[1]), [gc1/3]*3 + [gc2/6]*6 + [0.2], list(alpha) + ['N/A'], colors):
    ax.text(x=x-0.2, y=w, s=f'{np.round(w*100,1)}%', color=c, ha='right', va='center', fontsize=6)

#add label legend
lofinal = r'$\mathrm{\mathsf{   _{final}   }}$'
loinput = r'$\mathrm{\mathsf{   _{input}   }}$'
ax.text(x=0.7, y=0.7, s='Mean Weight', ha='right', va='center', fontsize=8, transform=ax.transAxes, color='black')
ax.scatter(x=0.725, y=0.7, marker='_', s=200, color='black',transform=ax.transAxes)

#format
xmin, xmax = ax.get_xlim()
ax.hlines(y=0, xmin=xmin, xmax=xmax, color='black', linewidth=1.0)
# ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (c) }$" + ' Sampled kernel weights', ha='left', va='top', fontsize=10, transform=ax.transAxes)

ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(0,);



####################### TOP RIGHT PLOT ######################################################
ax = axes[0,1]
Ws = []
Y = []
X1 = []
EV = []
target = 18
# run analysis
for _ in range(nruns):
    # plot the known kernels
    Wrun = np.concatenate([np.random.dirichlet(alpha[:3])*gc1, np.random.dirichlet(alpha[3:])*gc2])
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        if _ == 0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)
        ax.plot(xplot, yplot, linestyle='--', linewidth=0.05, color=color)

    # plot the phantom kernel
    ev = sum(X*Wrun)/sum(Wrun)
    EV.append(ev)
    xnew = target + (target-ev)/sim*rep
    X1.append(xnew)
    yplot = norm.pdf(xplot, xnew, bw_base)*sim
    if _ == 0:
        label = f'Phantom Kernel ({int(np.round(sim*100,2))}%)'
        ax.plot(xplot, [0]*len(xplot), linestyle='-', linewidth=0.5, color='purple', zorder=0, label=label)
    else:
        label = ''
        ax.plot(xplot, yplot, linestyle='-', linewidth=0.04, color='purple', zorder=3, label=label, alpha=0.5)

    ytotal += yplot
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=0.01, color='black', alpha=0.5)

    Xnew = np.concatenate([X,[xnew]])
    Wnew = np.concatenate([Wrun,[sim]])
    ev = sum(Xnew*Wnew)/sum(Wnew)
    Ws.append(Wnew)
    Y.append(ytotal)
# format arrays
Y = np.array(Y)
Ws = np.array(Ws)

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
aiqr = np.trapz(qhi, xplot) - np.trapz(qlo, xplot)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f"A{loiqr}")
ax.text(0.95, 0.95, f"A{loiqr}={'%.2f' % aiqr}", ha='right', va='top', fontsize=8, transform=ax.transAxes)
print('A_IQR')
print(aiqr)
print('')

# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean PDF')

#format
ax.hlines(y=0, xmin=np.min(xplot), xmax=np.max(xplot), color='black', linewidth=1.0)
ymin, ymax = ax.get_ylim()
ax.vlines(x=EV, ymin=0, ymax=ymax*0.95, alpha=0.01, color=color_evold)
ax.text(x=np.mean(EV), y=ymax*0.96, s=f'EV{looriginal}', ha='right', va='bottom', fontsize=10, color=color_evold)
ax.vlines(x=target, ymin=0, ymax=ymax*0.95, linewidth=1, color='black')
ax.text(x=target, y=ymax*0.95, s=f'EV{lotarget}', ha='left', va='bottom', fontsize=10, color='black')

ax.text(x=X[2], y=-0.025, s=f'Group 1 = {int(gc1*100)}%', ha='right', va='top', fontsize=8)
ax.text(x=X[3], y=-0.025, s=f'Group 2 = {int(gc2*100)}%', ha='left', va='top', fontsize=8)

ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
# ax.set_ylim(0,);
xleg = 0.75
# ax.legend(loc='upper left', bbox_to_anchor=(xleg, 1), fontsize=8);
ax.text(x=0, y=ytitle, s=r"$\bf{ (b) }$" + f' Using phantom kernels to achieve ' + r"$\bf{ much }$" f' higher EV{lotarget}', ha='left', va='top', fontsize=10, transform=ax.transAxes)


# dimension labels below plot
(left, bottom, width, height) = (0.5, 0.35, 0.4, 0.15)
ax = plt.axes([left, bottom, width, height], facecolor=(1,1,1,0)) #[left,  bottom, width, height]
lw = 0.5
ax.set_xticks([])
ax.set_yticks([])
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

x1, x2, x3 = (0.36, 0.43, 0.73)
yline = 0.5
xnotch = 0.01
ynotch = xnotch*width/height*2

ax.hlines(y=yline, xmin=x1, xmax=x3, color='black', transform=ax.transAxes, linewidth=lw)
ax.plot([x1-xnotch, x1+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)
ax.plot([x2-xnotch, x2+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)
ax.plot([x3-xnotch, x3+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)

ax.text(x1, yline+ynotch, s=f'EV{looriginal}', va='bottom', ha='right', transform=ax.transAxes, fontsize=8)
ax.text(x2, yline+ynotch, s=f'EV{lotarget}', va='bottom', ha='center', transform=ax.transAxes, fontsize=8)
ax.text(x3, yline+ynotch, s='Phantom\nKernel', va='bottom', ha='center', transform=ax.transAxes, fontsize=8)

ax.text(x2-2*xnotch, yline-4*ynotch, s=f'EV{lotarget}-EV{looriginal}', va='top', ha='right', transform=ax.transAxes, fontsize=8)
ax.text(x2+2*xnotch, yline-4*ynotch, s=f'(EV{lotarget}-EV{looriginal}) '+r'$\frac{rep}{1-rep}$', va='top', ha='left', transform=ax.transAxes, fontsize=8)

# for vline in np.arange(1,10)/10:
#     ax.vlines(vline, ymin, ymax, color='red', transform=ax.transAxes, linewidth=lw)

# for hline in np.arange(1,10)/10:
#     ax.hlines(hline, xmin, xmax, color='red', transform=ax.transAxes, linewidth=lw)




####################### FORMAT ######################################################

ax = axes[0,0]
ax.text(x=0, y=ytitle+0.15, s='Scenario 4.1', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)

ax = axes[0,1]
ax.text(x=0, y=ytitle+0.15, s='Scenario 4.2', ha='left', va='bottom', fontsize=12, transform=ax.transAxes);

ax = axes[1,0]


fig.delaxes(axes[1,1])

plt.savefig('outputs/figures/SCENARIO4_IntroduceEVTarget.jpg', bbox_inches='tight')

## Multi variate analysis

In [ ]:
%%time
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
bw_base = 0.9*np.std(X)*len(X)**-0.2
# xplot = np.linspace(np.max([0,np.min(X)-10*bw_base]), np.min(X)+20*bw_base, 1_001)
xplot = np.linspace(0, 50, 1_001)


MEAN = np.array([1/4, 2/4, 3/4, 4/4, 5/4])*np.mean(X)
STD = np.array([0/4, 1/4, 2/4, 3/4, 4/4])*np.std(X)
REP = np.arange(1,10)/10

# MEAN = np.array([0.75, 1.0, 1.25])*np.mean(X)
# STD = np.array([0, 0.5, 1])*np.std(X)
# REP = np.arange(1,4)/4
# REP = np.arange(1,10)/10


results_dict = {}

for (m, mean), (s, std), (r, rep) in product(enumerate(MEAN), enumerate(STD), enumerate(REP)):
    string = f'm{m}s{s}r{r}'
    evtargets = np.random.normal(mean, std, 1000)
    xplot, yplot, dict_analytics = kl2(data=X, xplot=xplot, evtargets=evtargets, represented=rep, nruns=1000, progressbar=False)
    
    results_dict[string] = {
        'xplot': xplot,
        'yplot': yplot,
        'area_iqr': find_area_iqr(xplot, yplot),
        'dict_analytics': dict_analytics,
    }

In [ ]:
%%time
fontsize = 12
colors = sns.color_palette('flare', len(REP))
fig, axes = plt.subplots(len(MEAN), len(STD), figsize=(10,6), gridspec_kw=dict(hspace=0.1, wspace=0.1), sharex=True)
for (col, mean), (row, std) in product(enumerate(MEAN), enumerate(STD)):
    m = col
    s = row
    ax = axes[row, col]
    
    # main plots
    for (r, rep) in enumerate(REP):
        string = f'm{m}s{s}r{r}'
        xplot = results_dict[string]['xplot']
        yplot = results_dict[string]['yplot']
        area_iqr = results_dict[string]['area_iqr']
        ax.plot(xplot, yplot.mean(axis=1), color=colors[r], linewidth=area_iqr)
    
    #supplementary plots
    for color, x in zip(sns.color_palette('tab10', len(X)), X):
        ax.scatter(x, 0, marker='|', s=100, color=color)
    ax.fill_between(xplot, norm.pdf(xplot, mean, std)*0.1, linewidth=0, color='tab:blue', alpha=0.5, label=f'EV{lotarget} Distribution')
    
    #formatting
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_xticks([])
    ax.set_ylim(0,)
    ax.set_xlim(0,)

    # ax.legend()
    if col == 0:
        ax.text(-0.1, 0.5, f'{np.round(std/np.std(X),1)}σ{lodata}', ha='right', va='center', transform=ax.transAxes, fontsize=fontsize)
    if row == 0:
        ax.text(0.5, 1.1, f'{np.round(mean/np.mean(X),2)}μ{lodata}', ha='center', va='bottom', transform=ax.transAxes, fontsize=fontsize)
    
ax = axes[2,0]
ax.text(x=-0.7, y=0.5, s=f'EV{lotarget} StDev', rotation=90, ha='center', va='center', transform=ax.transAxes, fontsize=fontsize+2)

ax = axes[0,2]
ax.text(x=0.5, y=1.5, s=f'EV{lotarget} Mean', rotation=0, ha='center', va='bottom', transform=ax.transAxes, fontsize=fontsize+2)

#legend
ax = axes[0, len(MEAN)-1]
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
for rep, color in zip(REP, colors):
    ax.plot(xmin, ymin, color=color, label=f'{int(np.round(rep*100,0))}%', linewidth=2)
handles, labels = ax.get_legend_handles_labels()
# sort both labels and handles by labels
labels, handles = zip(*sorted(zip(labels, handles), key=lambda t: t[0]))
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1,1))
ax.text(1.025,1,'Representativeness', fontsize=fontsize, ha='left', va='bottom', transform=ax.transAxes)
# ax.legend(loc='upper left', bbox_to_anchor=(1,1))

#legend for line thickness
(left, bottom, width, height) = (0.91, 0.3, 0.18, 0.15)
ax = plt.axes([left, bottom, width, height], facecolor=(1,1,1,0)) #[left,  bottom, width, height]
lw = 0.5
ax.set_xticks([])
ax.set_yticks([])
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)


areas = []
for string in results_dict:
    areas.append(results_dict[string]['area_iqr'])
areas = np.array(areas)
lws = np.linspace(np.round(np.min(areas),1), np.round(np.max(areas),1), 5)

for i, lw in enumerate(lws):
    y = 1-(i+0.5)/len(lws)
    ax.hlines(y, 0, 0.2, color='black', linewidth=lw, transform=ax.transAxes)
    ax.text(0.23, y, np.round(lw, 1), ha='left', va='center', transform=ax.transAxes, fontsize=10)

ax.text(0.23, 1, f'A{loiqr}', ha='left', va='bottom', transform=ax.transAxes, fontsize=10)
# yline = 0.5
# xnotch = 0.01
# ynotch = xnotch*width/height*2

# ax.hlines(y=yline, xmin=x1, xmax=x3, color='black', transform=ax.transAxes, linewidth=lw)
# ax.plot([x1-xnotch, x1+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)
# ax.plot([x2-xnotch, x2+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)
# ax.plot([x3-xnotch, x3+xnotch], [yline-ynotch, yline+ynotch], color='black', transform=ax.transAxes, linewidth=lw)

# ax.text(x1, yline+ynotch, s=f'EV{looriginal}', va='bottom', ha='right', transform=ax.transAxes, fontsize=8)
# ax.text(x2, yline+ynotch, s=f'EV{lotarget}', va='bottom', ha='center', transform=ax.transAxes, fontsize=8)
# ax.text(x3, yline+ynotch, s='Phantom\nKernel', va='bottom', ha='center', transform=ax.transAxes, fontsize=8)

# ax.text(x2-2*xnotch, yline-4*ynotch, s=f'EV{lotarget}-EV{looriginal}', va='top', ha='right', transform=ax.transAxes, fontsize=8)
# ax.text(x2+2*xnotch, yline-4*ynotch, s=f'(EV{lotarget}-EV{looriginal}) '+r'$\frac{rep}{1-rep}$', va='top', ha='left', transform=ax.transAxes, fontsize=8)

# for vline in np.arange(1,10)/10:
#     ax.vlines(vline, ymin, ymax, color='red', transform=ax.transAxes, linewidth=lw)

# for hline in np.arange(1,10)/10:
#     ax.hlines(hline, xmin, xmax, color='red', transform=ax.transAxes, linewidth=lw)




plt.savefig('outputs/figures/MultivariateAnalysis.jpg', bbox_inches='tight')

In [ ]:
# # where are the phantom kernels?
# XPK = []
# reps = []
# means = []
# area_iqrs = []
# for string in results_dict:
#     xpk = np.mean(results_dict[string]['dict_analytics']['all_x'][:,-1])
#     xpk = results_dict[string]['dict_analytics']['all_x'][:10,-1]
#     XPK.append(xpk)
#     rep = (float(string[string.find('r')+1])+1)/10
#     reps.append(np.zeros_like(xpk)+rep)
#     mean = (float(string[string.find('m')+1])+1)/10
#     means.append(np.zeros_like(xpk)+mean)
#     area_iqr = results_dict[string]['area_iqr']
#     area_iqrs.append(np.zeros_like(xpk)+area_iqr)

# XPK = np.array(XPK).flatten()
# reps = np.array(reps).flatten()
# means = np.array(means).flatten()
# area_iqrs = np.array(area_iqrs).flatten()

# fig, ax = plt.subplots()

# plt.scatter(reps, XPK, c=area_iqrs, s=1)
# ax.set_xlabel('reps')
# ax.set_ylabel('phantom kernel')
# ax.text(1.05, 1.05, 'Uncertainty', ha='left', va='bottom', transform=ax.transAxes)
# plt.colorbar()

In [ ]:
columns = [f'EV{lotarget} Mean', f'EV{lotarget} StDev', 'Representativeness', f'A{loiqr}']
dft = pd.DataFrame(columns=columns)
for string in results_dict:    
    m = MEAN[int(string[string.find('m')+1])]
    s = STD[int(string[string.find('s')+1])]
    r = REP[int(string[string.find('r')+1])]
    area_iqr = results_dict[string]['area_iqr']

    dft.loc[len(dft)] = [m, s, r, area_iqr]


fig, ax = plt.subplots(figsize=(6,4))
x = 'Representativeness'
y = f'A{loiqr}'
s = f'EV{lotarget} StDev'
c = f'EV{lotarget} Mean'
alpha = 0.5
# shapes = ['^', 's', 'p', 'h', 'o']
# markers = (dft['Mean'].map({k:v for k, v in zip(MEAN, shapes)})).values
# for cat in dft['Mean'].unique():
#     dft2 = dft.loc[dft['Mean'] == cat]
#     plt.scatter(dft2[x], dft2[y], c=dft2[c], marker=markers_dict[cat])
plt.scatter(dft[x], dft[y], c=dft[c]/np.mean(X), s=dft[s].rank(method='dense')*20, alpha=alpha, cmap='crest')

#legend
ylegend = np.linspace(0.75,0.95,5)
xlegend = np.array([0.7]*len(ylegend))
ax.scatter(xlegend, ylegend, s=(dft[s].rank(method='dense')*20).unique(), color='black', alpha=alpha, transform=ax.transAxes)
legendlabels = [f'%.2f' % ele for ele in dft[s].unique()/np.std(X)]
legendlabels[-1] = f'{legendlabels[-1]} σ{lodata}'
for xleg, yleg, text in zip(xlegend, ylegend, legendlabels):
    ax.text(xleg+0.05, yleg, text, ha='left', va='center', transform=ax.transAxes)
ax.text(xlegend[0], np.max(ylegend)+0.05, s, transform=ax.transAxes)

#format
ticks = (dft[c]/np.mean(X)).unique()
cbar = plt.colorbar(ticks=(dft[c]/np.mean(X)).unique())
ticks = [f'%.2f' % ele for ele in ticks]
ticks[-1] = f'{ticks[-1]}μ{lodata}'
cbar.ax.set_yticklabels(ticks)
ax.text(1.05, 1.05, c, ha='left', va='bottom', transform=ax.transAxes)
ax.set_xlabel(x)
ax.set_ylabel(y)
ax.spines[['top', 'right']].set_visible(False)

plt.savefig('outputs/figures/ResultsFromMultivariateAnalysis.jpg', bbox_inches='tight')

In [ ]:
# xtest = np.linspace(0,10,50)
# ytest = norm.pdf(xtest, 5, 3)

# fig, ax = plt.subplots()
# markers = ['^', 's', 'p', 'h', 'o']
# for i, marker in enumerate(markers):
#     ax.scatter(xtest, ytest*(10-i)/10, marker=marker);

# RESULTS - STEEL

## Import data

In [ ]:
#steel EPDs
filepath = 'data/EPDs_steelstructural_2025-01-09.xlsx'
feat = 'manufacturer : country'
df = pd.read_excel(filepath)
df = df.loc[~df['ecc'].isna()]
df = df.loc[~df[feat].isna()]
df = df.set_index('open_xpd_uuid')
data = df['ecc'].values
countries = df[feat].values
print(f'{len(data)} EPDs from {len(set(countries))} countries')

In [ ]:
feat = 'category : display_name'
df[feat].value_counts()

In [ ]:
#steel production quantities

#pull in country codes
filepath = 'data/CountryCodes.xlsx'
dfcc = pd.read_excel(filepath)
dct_cc = {}
for row in dfcc.index:
    country = dfcc.loc[row, 'STEEL-COUNTRY']
    code = dfcc.loc[row, 'ALPHA-2']
    if type(country) == str and type(code) == str:
        dct_cc[country] = code
        dct_cc[code] = country

#pull in steel production data
filepath = 'data/WorldSteelReport_exp-2025-03-14_17_27_36.xlsx'
df_prod = pd.read_excel(filepath, skiprows=2)
df_prod = df_prod.loc[0:list(df_prod['Country']).index('Yemen')]
# srs = df_prod.iloc[:,1:].sum(axis=1)
# df_prod = df_prod.iloc[~srs.loc[srs == 0].index]
df_prod.set_index('Country', inplace=True)
df_prod = df_prod.astype(float)
df_prod['Total'] = df_prod.sum(axis=1)
df_prod['Proportion'] = df_prod['Total']/df_prod['Total'].sum()*2 #multiply by two because 'World' is its own row
df_prod.sort_index(inplace=True)

In [ ]:
# two axis bar plot to visualize EPDs and tons of steel production

lst = [dct_cc[cc] for cc in set(countries)]
df_prod_filtered = df_prod.loc[lst, :]
df_prod_filtered = df_prod_filtered.loc[df_prod_filtered['Total'].sort_values(ascending=True).index].iloc[:,:-2]
df_prod_filtered /= 1000
width = 0.4
bbox_to_anchor = (0.7, 0.3)


fig, ax = plt.subplots(figsize=(6,6))

# first plot
# dft.plot(kind='barh', align='edge', stacked=True, ax=ax, width=-width)
df_prod_filtered.sum(axis=1).plot(kind='barh', align='edge', stacked=True, ax=ax, width=-width, label='Steel Production')
ax.set_xlabel('Million Tonnes of Steel Production', fontsize=10)
ax.set_ylabel('')
ax.tick_params(axis='y', which='major', labelsize=10)

for y, (country, x) in enumerate(df_prod_filtered.sum(axis=1).items()):
    perc = df_prod['Proportion'][country]
    perc = f'{np.round(perc*100,1)}%'
    ax.text(x=x+5, y=y-width/2, s=perc, va='center', ha='left', color='tab:blue', fontsize=8)

ax.legend(bbox_to_anchor=bbox_to_anchor, loc='upper left', fontsize=8, frameon=False)




# second plot
feat = 'manufacturer : country'
color = sns.color_palette('tab10', 6)[5]
ax2 = ax.twiny()
ind = [dct_cc[ele] for ele in df[feat].value_counts().index]
df_nepd = pd.DataFrame(index=ind, data=df[feat].value_counts().values, columns=['EPD_count'])

df_nepd = df_nepd.loc[df_prod_filtered.index,:]

df_nepd.plot(kind='barh', ax=ax2, align='edge', color='tab:orange', alpha=1.0, width=width, )
ax2.set_xlabel('Number of ECCs', fontsize=10)
ax2.set_ylabel('')
ax2.legend(bbox_to_anchor=bbox_to_anchor, loc='lower left', fontsize=8, frameon=False, labels=['ECC Count'])

for y, (country, x) in enumerate(df_nepd['EPD_count'].items()):
    ax2.text(x=x+0.5, y=y+width/2, s=x, va='center', ha='left', color='tab:orange', fontsize=8)

#format
for a in [ax, ax2]:
    a.spines[['top', 'right', 'bottom']].set_visible(False)

plt.savefig('outputs/figures/EPDCount_vs_SteelProduction_v1.jpg', bbox_inches='tight')

In [ ]:
# prep data
x = df_prod_filtered.sum(axis=1)
y = df_nepd['EPD_count']

# plot
fig, ax = plt.subplots()
ax.scatter(x, y, s=2)



# add labels
xlo, xhi = ax.get_xlim()
ylo, yhi = ax.get_ylim()
xrng = xhi-xlo
yrng = yhi-ylo
pad = 1.025
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for i, label in enumerate(df_prod_filtered.index):
        ha = 'left'
        va = 'center'
        xytext = (x[i]*pad, y[i])
        rotation=0
    #     if label in ['Luxembourg', 'Sweden', 'United Arab Emirates']:
    #         ha = 'right'

    #     elif label in ['Italy', 'Türkiye']:
    #         va = 'bottom'
    #         ha = 'left'

    #     elif label in ['Taiwan']:
    #         ha='left'

    #     elif label in ['Indonesia']:
    #         ha='left'

    #     elif label in ['Netherlands', 'Australia']:
    #         va='bottom'
    #         ha='left'
    #         rotation = 25

    #     elif label in ['Spain']:
    #         va='bottom'
    #         ha='left'
    #         rotation = 15

    #     elif label in ['United States', 'Japan']:
    #         va='bottom'

    #     elif label in ['Czechia']:
    #         ha='right'
    #         va = 'top'
    #         rotation = 20

    #     elif label in ['Canada']:
    #         ha='left'
    #         va='top'
    #         rotation=-10

    #     elif label in ['United Kingdom']:
    #         ha='left'
    #         va='top'
    #         rotation=-20

    #     elif label in ['Slovakia']:
    #         ha='left'
    #         va='top'
    #         rotation=-30
        if label == 'Czechia':
            xytext = (x[i]/pad, y[i])
            ha = 'right'

        elif label == 'Canada':
            xytext = (x[i]*pad, y[i]*pad)

            ha = 'center'
            va = 'bottom'

        elif label == 'Netherlands':
            xytext = (x[i]*pad, y[i]*pad)
            va = 'bottom'

        elif label == 'United Kingdom':
            xytext = (x[i]*pad, y[i]/pad)
            va = 'top'


        plt.annotate(label, (x[i], y[i]), ha=ha, va=va, xytext=xytext, fontsize=6, rotation=rotation)

# format
ax.spines[['top', 'right']].set_visible(False)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Million Tonnes of Steel Production, 2020-2024')
ax.set_ylabel('Number of Steel EPDs')

plt.savefig('outputs/figures/EPDCount_vs_SteelProduction_v2.jpg', bbox_inches='tight')

In [ ]:
country = 'United Kingdom'
print(country)
print(x[country], ', ', y[country])
print('')

country = 'Netherlands'
print(country)
print(x[country], ', ', y[country])
print('')

## Perform analysis with steel production data

In [ ]:
%%time
# get group constraints and represented
group_constraints = {}
represented = 0
for cc in set(countries):
    country = dct_cc[cc]
    key = tuple([i for i, country in enumerate(countries) if country == cc])
    prop = df_prod.loc[country, 'Proportion']
    represented += prop
    group_constraints[key] = prop

represented = sum(group_constraints.values())
    
# define target
ia_epds = np.array([1.850, 1.870, 0.603, 1.220, 1.170, 1.190, 1.730, 1.710, 1.730, 1.720, 1.990, 2.780])
ia_epds = np.array([1.850, 1.870, 1.220, 1.170, 1.190, 1.730, 1.710, 1.730, 1.720, 1.990]) #outliers removed
ia_epds_kde = scipy.stats.gaussian_kde(ia_epds, bw_method='silverman').resample(1000)
reall = 0

#set xplot
xplot = np.linspace(0,7.0,1001)
nruns = 1_000

#run analysis

xplot, yplot, dict_analytics = kl2(data=data, xplot=xplot, group_constraints=group_constraints, BW_factors=None,
                              evtargets=ia_epds_kde, represented=None, nruns=nruns)




In [ ]:
%%time
fig, ax = plt.subplots(figsize=(8,4))
kl2_plot(xplot, yplot, ax, lo=0.25, hi=0.75, color='tab:red', include_area_iqr=False)
#plot standard KDE
alpha = np.zeros_like(data).astype(float)
for group, perc in group_constraints.items():
    for ind in group:
        alpha[ind] += perc/len(group)
alpha /= min(alpha)
bw = weighted_bw(data, alpha)
ykde = np.zeros_like(xplot)
for x, w in zip(data, alpha):
    ykde += norm.pdf(xplot, x, bw)*w/np.sum(alpha)
ax.plot(xplot, ykde, label='Standard Weighted KDE', color='tab:blue')
# sns.kdeplot(data, ax=ax, weights=alpha, label='Standard KDE')


color = 'tab:purple'

#formatting
ax.scatter(ia_epds, [0.03]*len(ia_epds), marker='|', color=color, s=200, label=f'EV{lotarget}', zorder=1, linewidth=0.75)
ax.scatter(data, [0.015]*len(data), marker='|', color='tab:blue', s=100, label=f'ECCs', zorder=2, linewidth=0.25)
ax.legend(bbox_to_anchor=(1,0.025), loc='lower right')
ax.set_xlim(0,)
ax.set_ylim(0,)
ax.set_xlabel(f'Steel ECC (kgCO{lo2}e/kg)', fontsize=14)

#labels
x=0.5
y=0.85
ydiff = 0.06
fontsize=12
ax.text(x=x, y=y, s=f'Representativeness={int(np.round(represented*100,0))}%', ha='left', va='bottom', transform=ax.transAxes, fontsize=fontsize)
y -= ydiff
area_iqr = find_area_iqr(xplot, yplot)
ax.text(x=x, y=y, s=f"A{loiqr}={'%.2f' % np.round(area_iqr,2)}", ha='left', va='bottom', transform=ax.transAxes, fontsize=fontsize);

plt.savefig('outputs/figures/ProofOfConcept_GlobalSteel.jpg', bbox_inches='tight')


# _
# _
# Scratchpad

In [ ]:
# showing how the uncertainty metric works
X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
bw_base = 0.9*np.std(X)*len(X)**-0.2
xplot = np.linspace(np.min(X)-3*bw_base, np.max(X)+3*bw_base, 1_001)
ytotals1 = []
ytotals2 = []

for _ in range(1000):
    W = np.random.dirichlet(np.ones_like(X))
    yi = norm.pdf(xplot.reshape(-1,1), X, bw_base)*W
    ytotals1.append(yi.sum(axis=1))
    
    W = np.random.dirichlet(np.ones_like(X)*10)
    yi = norm.pdf(xplot.reshape(-1,1), X, bw_base)*W
    ytotals2.append(yi.sum(axis=1))

ytotals1 = np.array(ytotals1)
ytotals2 = np.array(ytotals2)

In [ ]:
# showing how the uncertainty metric works
fig, axes = plt.subplots(2,1, figsize=(6,4))

for i, yplot in enumerate([ytotals1.T, ytotals2.T]):
    ax = axes[i]
    qlo = np.quantile(yplot, 0.25, axis=1)
    qhi = np.quantile(yplot, 0.75, axis=1)
    ax.plot(xplot, yplot, linewidth=0.01, color='black');
    ax.plot(xplot, yplot.mean(axis=1), linewidth=2, color='black')
    ax.plot(xplot, qlo, color='tab:red', linestyle='--')
    ax.plot(xplot, qhi, color='tab:red', linestyle='--')
    ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=0.5)
    
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_xticks([])
    ax.set_ylim(0,)
    
    area_iqr = np.round(np.trapz(qhi, xplot) - np.trapz(qlo, xplot), 3)
    ax.text(1,1,f'Uncertainty Metric = {area_iqr}', ha='right', va='top', transform=ax.transAxes)
    

In [ ]:
# xplot1, yplot1, dict_analytics1 = kl2(data=X, xplot=xplot, evtargets=[13])
# xplot2, yplot2, dict_analytics2 = kl2(data=X, xplot=xplot, evtargets=np.random.uniform(10,30,100))

# fig, axes = plt.subplots(2,1)

# ax = axes[0]
# kl2_plot(xplot1, yplot1, ax)

# ax = axes[1]
# kl2_plot(xplot2, yplot2, ax)

# Playing with the Dirichlet distribution

In [ ]:
%%time
maxes = [1, 10, 100, 1000]
n = 10_000
allx = []
ally = []
fig, axes = plt.subplots(len(maxes),1, sharey=True, gridspec_kw=dict(hspace=0.5))
for i, xmax in enumerate(maxes):
    ax = axes[i]
    xplot = []
    yplot = []
    for _ in range(10000):
        x = np.random.uniform(0,xmax,10)
        y = np.random.dirichlet(x)
        xplot += list(x)
        yplot += list(y)

    if i == 0:
        allx += xplot
    ax.scatter(xplot, yplot, s=0.001)
    ax.set_ylim(0,)
    ax.get_yaxis().set_visible(False)
    
fig.suptitle('Exploring the effect of alpha on the Dirichlet distribution')

In [ ]:
sizes = [3, 5, 10, 25]

fig, axes = plt.subplots(len(sizes),1)

for i, size in enumerate(sizes):
    ax = axes[i]
    xplot = np.arange(1,size+1)/size
    yplot = []
    for _ in range(1000):
        yplot.append(np.random.dirichlet(xplot))
    yplot = np.array(yplot)

    sns.boxplot(yplot, color='white', fliersize=0, ax=ax)
    sns.stripplot(yplot, s=1, ax=ax)
    ax.set_xticklabels(xplot, fontsize=6)

fig.suptitle('How does likelihood change with bigger alpha, normalized to one?')

In [ ]:
xplots = [[0.25,0.5,0.75,1],[0.2,0.4,0.6,1],[0.1,0.2,0.3,1],[0.01,0.02,0.03,1]]

fig, axes = plt.subplots(len(xplots),1)

for i, xplot in enumerate(xplots):
    ax = axes[i]
    yplot = []
    for _ in range(1000):
        yplot.append(np.random.dirichlet(xplot))
    yplot = np.array(yplot)

    sns.boxplot(yplot, color='white', fliersize=0, ax=ax)
    sns.stripplot(yplot, s=1, ax=ax)
    ax.set_xticklabels(xplot, fontsize=6)

fig.suptitle("What if there's one alpha that's much higher than the others?")

# Compare KDE2 with DPGMM

# Demonstrating that the mean of a distribution is the same as selecting randomly

In [ ]:
#how to pull random samples from a PDF represented by X and Y values
def random_samples(X, Y, n=1):
    pdf = Y/scipy.integrate.trapezoid(Y, X)
    cdf = np.cumsum(pdf) * np.diff(X, prepend=X[0])
    inverse_cdf = scipy.interpolate.interp1d(cdf, X, kind='linear', fill_value="extrapolate")
    return inverse_cdf(np.random.uniform(0,1,n))


fig, ax = plt.subplots()
ax.plot(xplot, yplot[0])
ax.hist(random_samples(xplot, yplot[0], n=10_000), density=True, bins=50);

In [ ]:
# demonstrating that the mean yplot best reflects randomly sampling a distribution then randomly sampling from that distribution

fig, ax = plt.subplots()

# real samples
samples = []
for _ in range(100_000):
    i = random.randint(0,yplot.shape[0]-1)
    samples.append(random_samples(xplot, yplot[i], n=1))
samples = np.array(samples).flatten()
ax.hist(samples, bins=100, density=True, label='true data');

#by median
q50 = np.quantile(yplot, 0.50, axis=0)
q50 /= scipy.integrate.trapezoid(q50, xplot)
ax.plot(xplot, q50, label='median');

#by mean
means = np.mean(yplot, axis=0)
print(scipy.integrate.trapezoid(means, xplot))
means /= scipy.integrate.trapezoid(means, xplot)
print(scipy.integrate.trapezoid(means, xplot))
ax.plot(xplot, means, label='mean');

ax.legend()

# Old Scenario 2.1 and 2.2 with alpha=ones and alpha=fours and checking similarity of results

In [ ]:
# OLD SCENARIO 2.1 and 2.2 with alpha=ones and alpha=fours
%%time


X = np.array([11, 12, 13, 16, 17, 18, 19, 20, 21])
BW_factors = np.array([0.88247698, 1.05838405, 1.08353463, 0.99264064, 0.98733438,
       0.89708815, 0.91940053, 1.04995333, 1.12918731])
BW_factors = BW_factors / np.mean(BW_factors)
alpha = np.ones_like(X)
W = alpha/np.sum(alpha)
bw_base = weighted_bw(X, W, bw_method='silverman')
BW = BW_factors*bw_base
std_base = weighted_std(X, W)
xplot = np.linspace(min(X)-std_base*3, max(X)+std_base*3, 1_001)
colors = sns.color_palette('tab10', len(X))

nrows = 2
ncols = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(12,4),
                         gridspec_kw=dict(hspace=0.75, wspace=0.1))
ytitle = 1.2
nruns = 1_000
lw = 0.01

alpha_aiqr = 1.0

####################### TOP LEFT PLOT ######################################################
ax = axes[0,0]

Ws = []
Y = []
# run analysis
for _ in range(nruns):
    Wrun = np.random.dirichlet(alpha)
    Ws.append(Wrun)
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
        if _==0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
    Y.append(ytotal)

# format arrays and save results
Y = np.array(Y)
Ws = np.array(Ws)
yplot_s21 = Y

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f'A{loiqr}')
print('A_IQR')
print(np.trapz(qhi, xplot) - np.trapz(qlo, xplot))
print('')

# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean Resulting PDF')

#format
ax.text(x=0, y=ytitle, s=r"$\bf{ (a) }$" + f' Resulting PDFs when all α={int(alpha[0])}', ha='left', va='top', fontsize=10, transform=ax.transAxes)
ax.legend(loc='upper left', bbox_to_anchor=(0,1), fontsize=8)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_xlabel(f'Embodied Carbon Coefficient (kgCO{lo2}e/unit)')

# ytmin, ytmax = ax.get_ylim()
ax.set_ylim(0, 0.2)


####################### BOTTOM LEFT PLOT ######################################################
ax = axes[1,0]

#plot
sns.stripplot(Ws, ax=ax, size=0.5)
ax.scatter(np.arange(len(alpha)), alpha/sum(alpha), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, y, s, c in zip(np.arange(len(alpha)), alpha/sum(alpha), alpha, colors):
    ax.text(x=x-0.2, y=y, s=s, color=c, ha='right', va='bottom', fontsize=6)
    ax.text(x=x-0.2, y=y, s=f'{np.round(s/sum(alpha)*100,1)}%', color=c, ha='right', va='top', fontsize=6)

#add label legend
y = 0.6
ax.text(x=0.2, y=y, s='α', ha='right', va='bottom', fontsize=8, transform=ax.transAxes, color='black')
ax.text(x=0.2, y=y, s='Mean Weight', ha='right', va='top', fontsize=8, transform=ax.transAxes, color='black')
ax.scatter(x=0.225, y=y, marker='_', s=200, color='black',transform=ax.transAxes)

#add BW labels

#format
# ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (b) }$" + ' Sampled kernel weights of (a)', ha='left', va='top', fontsize=10, transform=ax.transAxes)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(0,);
ybmin, ybmax = ax.get_ylim()


####################### TOP RIGHT PLOT ######################################################
ax = axes[0,1]

alpha = np.ones_like(X)*4
Ws = []
Y = []
for _ in range(nruns):
    Wrun = np.random.dirichlet(alpha)
    Ws.append(Wrun)
    ytotal = np.zeros_like(xplot)
    BW = bw_dirichlet(X, alpha, BW_factors, Wrun, bw_method='silverman')
    for x, w, bw, color in zip(X, Wrun, BW, colors):
        yplot = norm.pdf(xplot, x, bw)*w
        ytotal += yplot
        ax.plot(xplot, yplot, linestyle='-', linewidth=lw, color=color)
        if _==0:
            ax.scatter(x, -0.01, marker='|', color=color, s=50)
    ytotal /= np.trapz(ytotal, xplot)
    ax.plot(xplot, ytotal, linestyle='-', linewidth=lw, color='black')
    Y.append(ytotal)

# format arrays and save results
Y = np.array(Y)
Ws = np.array(Ws)
yplot_s22 = Y

# plot A_IQR
qlo = np.quantile(Y, 0.25, axis=0)
qhi = np.quantile(Y, 0.75, axis=0)
ax.fill_between(xplot, qlo, qhi, color='tab:red', alpha=alpha_aiqr, label=f'A{loiqr}')
print('A_IQR')
print(np.trapz(qhi, xplot) - np.trapz(qlo, xplot))
print('')

# plot mean resulting PDF
ax.plot(xplot, np.mean(Y, axis=0), linestyle='-', linewidth=2, color='black', label='Mean Resulting PDF')

#format
ax.text(x=0, y=ytitle, s=r"$\bf{ (c) }$" + f' Resulting PDFs when all α={int(alpha[0])}', ha='left', va='top', fontsize=10, transform=ax.transAxes)

# ax.legend(fontsize=8)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
# ax.set_xlabel('Kernel Locations')
# ax.set_ylim(ytmin, ytmax)
ax.set_ylim(0, 0.2)


####################### BOTTOM RIGHT PLOT ######################################################
ax = axes[1,1]

#plot
sns.stripplot(Ws, ax=ax, size=0.5)
ax.scatter(np.arange(len(alpha)), alpha/sum(alpha), marker='_', s=200, color=colors)

#add labels to each strip plot
for x, y, s, c in zip(np.arange(len(alpha)), alpha/sum(alpha), alpha, colors):
    ax.text(x=x-0.2, y=y, s=s, color=c, ha='right', va='bottom', fontsize=6)
    ax.text(x=x-0.2, y=y, s=f'{np.round(s/sum(alpha)*100,1)}%', color=c, ha='right', va='top', fontsize=6)

#add label legend
# y = 0.6
# ax.text(x=0.2, y=y, s='α', ha='right', va='bottom', fontsize=8, transform=ax.transAxes, color='black')
# ax.text(x=0.2, y=y, s='Mean Weight', ha='right', va='top', fontsize=8, transform=ax.transAxes, color='black')
# ax.scatter(x=0.225, y=y, marker='_', s=200, color='black',transform=ax.transAxes)

#add BW labels

#format
# ax.set_title('(b) Sampled kernel weights based on α from (a)', loc='left')
ax.text(x=0, y=ytitle, s=r"$\bf{ (d) }$" + ' Sampled kernel weights of (c)', ha='left', va='top', fontsize=10, transform=ax.transAxes)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_xticks([]);
ax.set_ylim(ybmin, ybmax);


ax = axes[0,0]
ax.text(x=0, y=ytitle+0.15, s='Scenario 2.1', ha='left', va='bottom', fontsize=12, transform=ax.transAxes)
ax = axes[0,1]
ax.text(x=0, y=ytitle+0.15, s='Scenario 2.2', ha='left', va='bottom', fontsize=12, transform=ax.transAxes);

plt.savefig('outputs/figures/SCENARIO2_KernelsAndProductLevelUncertainty.jpg', bbox_inches='tight')

In [ ]:
# get results from Scenarios 2.1 and 2.2
results1 = []
results2 = []
for i in range(len(yplot_s22)):
    results1.append(effective_variance(xplot, yplot_s21[i])**0.5)
    results2.append(effective_variance(xplot, yplot_s22[i])**0.5)

fig, ax = plt.subplots(figsize=(6,2))
sns.stripplot(results1, orient='h', ax=ax, size=2, color='tab:blue')
# sns.kdeplot(results1, ax=ax, fill=True, color='tab:blue')
sns.boxplot(results1, ax=ax, orient='h', color='white', fliersize=0)

sns.stripplot(results2, orient='h', ax=ax, size=2, color='tab:orange')
# sns.kdeplot(results2, ax=ax, fill=True, color='tab:blue')
sns.boxplot(results2, ax=ax, orient='h', color='white', fliersize=0)


# double check results against manual KDE

bw_base = weighted_bw(X, alpha/sum(alpha), bw_method='silverman')
yplot_kde = np.zeros_like(xplot)
for x, w, bw_factor in zip(X, alpha/sum(alpha), BW_factors):
    yplot_kde += norm.pdf(xplot, x, bw_base*bw_factor)*w

ymin, ymax = ax.get_ylim()
ax.vlines(effective_variance(xplot, yplot_kde)**0.5, ymin, ymax, color='red')

In [ ]:

#Scenario 2.1 and 2.2 are very similar
fig, ax = plt.subplots()

#plot mean of each scenario
ax.plot(xplot, np.mean(yplot_s21, axis=0), label='S2.1', linewidth=2, linestyle='--', color='tab:orange')
ax.plot(xplot, np.mean(yplot_s22, axis=0), label='S2.2', linewidth=2, linestyle='-.', color='tab:green')

#plot all runs from each scenario
ax.plot(xplot, yplot_s21.T, linestyle='-', linewidth=0.01, color='tab:orange')
ax.plot(xplot, yplot_s22.T, linestyle='-', linewidth=0.01, color='tab:green')



yplot_kde = np.zeros_like(xplot)
for x, w, bw_factor in zip(X, W, BW_factors):
    yplot_kde += norm.pdf(xplot, x, bw_base*bw_factor)*w
ax.plot(xplot, yplot_kde, linewidth=2, color='tab:blue', label='manual kde')


ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_ylim(0,)
ax.set_xticks([])
ax.legend()
ax.set_title('Test plot used to visualize difference between \n resulting PDFs of different scenarios');

print('S21 StD: ', effective_variance(xplot, np.mean(yplot_s21, axis=0))**0.5)
print('S22 StD: ', effective_variance(xplot, np.mean(yplot_s22, axis=0))**0.5)
print('KDE StD: ', effective_variance(xplot, yplot_kde)**0.5)

In [ ]:
#checking to see how to calculate the necessary bandwidth to meet a certain standard deviation
STD_data = []
STD_result = []
colors = []
BW = []
Y = []

X = np.random.uniform(10,20,5)
W = np.random.dirichlet(np.ones_like(X))
BW_factors = np.random.uniform(0.9,1.1,5)
BW_factors = BW_factors / np.mean(BW_factors)

bw_base = weighted_bw(X, W, bw_method='silverman')
std_base_result = weighted_std(X, W, BW_factors*bw_base)
std_base_data = weighted_std(X, W)
xplot = np.linspace(min(X)-10*bw_base, max(X)+10*bw_base, 1_001)

print(f'{std_base_result} - Resulting StDev w/ KDE')
print(f'{std_base_data} - StDev from Data')

for _ in range(1_000):
    # StDev weighted
    Wrun = np.random.dirichlet(np.ones(len(X)))
    std_data = weighted_std(X, Wrun)
    STD_data.append(std_data)

    # Calculate what bw_run should be
    bw_run = weighted_bw(X, Wrun, bw_method='silverman')
    std_kernel = bw_run*BW_factors
    color = 'tab:green'
    var_data = std_data**2
    var_kernel = np.sum(Wrun*std_kernel**2)
    var_result = std_base_result**2
    if  var_data + var_kernel < var_result:
        color = 'tab:blue'
        bw_run *= np.sqrt((var_result - var_data) / var_kernel)
    
    #check if the function works - it does!!
    # test = np.round(bw_dirichlet(X, W, BW_factors, Wrun)/(bw_run*BW_factors), 4)
    # if np.max(test) > 1. or np.min(test) < 1.:
    #     print(f'Inequality: {bw_dirichlet(X, W, BW_factors, Wrun)} != {bw_run*BW_factors}')
    
    # Run KDE numerically to check if it worked
    ytotal = np.zeros_like(xplot)
    for x, w, bw_factor in zip(X, Wrun, BW_factors):
        ytotal += norm.pdf(xplot, x, bw_run*bw_factor)*w
    std_result = weighted_std(xplot, ytotal)
    STD_result.append(std_result)
    colors.append(color)
    Y.append(ytotal)
    BW.append(np.sqrt(np.sum(Wrun*(bw_run*BW_factors)**2)))

fig, axes = plt.subplots(2, 2, figsize=(6,6), gridspec_kw=dict(hspace=0.25))
################################################################################################
################################################################################################

ax = axes[0,0]
ax.set_title('NUMERICAL: std_data vs std_result', fontsize=8)
ax.scatter(STD_data, STD_result, c=list(colors), s=0.1)
ax.set_xlabel('StDev Run')
ax.set_ylabel('StDev KDE')

ax.set_ylim(0,)

ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()

ax.hlines(std_base_result, xmin, xmax, linestyle=':', color='gray')
ax.text(x=xmax, y=std_base_result, s='StDev Base (KDE)', va='top', ha='right', color='gray')

# ax.vlines(weighted_std(X, W), ymin, ymax, color='gray')
# ax.text(x=weighted_std(X, W), y=ymin, s='StDev Base (Weights)', va='bottom', ha='left', color='gray');

if ymax < std_base_result:
    ax.set_ylim(0, std_base_result*1.5)

################################################################################################
################################################################################################

ax = axes[0,1]
ax.set_title('Error: (Alg-Num)/Max', fontsize=8)
alg = np.sqrt(np.array(STD_data)**2 + np.array(BW)**2)
num = np.array(STD_result)
diff = (alg-num)/np.max([alg, num])
ax.scatter(np.array(STD_data), diff, s=0.1, color='purple')

ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()

ax.hlines(0, xmin, xmax, linestyle='-', color='black')
ax.text(xmin, ymax, 'Positive means Alg > Num', ha='left', va='top', fontsize=6)

################################################################################################
################################################################################################
ax = axes[1,0]
ax.set_title('ANALYTICAL: std_data vs std_result', fontsize=8)
ax.scatter(STD_data, alg, c=list(colors), s=0.1)
ax.set_xlabel('StDev Run')
ax.set_ylabel('StDev KDE')

ax.set_ylim(0,np.max(alg)*1.2)

ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()

ax.hlines(std_base_result, xmin, xmax, linestyle=':', color='gray')
ax.text(x=xmax, y=std_base_result, s='StDev Base (KDE)', va='top', ha='right', color='gray')

################################################################################################
################################################################################################
ax = axes[1,1]

Y = np.array(Y)
ax.plot(xplot, Y.T, color='black', linewidth=1, alpha=0.01);
ax.plot(xplot, Y.mean(axis=0), color='red');
ax.set_ylim(0, np.max(Y.mean(axis=0))*1.5)

In [ ]:
#double checking that algebraically and numerically calculating StDev of a plot get the same answer
for _ in range(10):

    X = np.random.uniform(10,15,5)
    W = np.random.dirichlet(np.ones(len(X)))

    mean = np.sum(X*W)
    var = np.sum(W * (X-mean)**2)
    std_base = weighted_std(X, W)
    iqr_base = weighted_quantile(X, W, 0.75, 'perc2val') - weighted_quantile(X, W, 0.25, 'perc2val')
    bw = weighted_bw(X, W, bw_method='silverman')
    std_alg = np.sqrt(var + bw**2)

    # print(f'{std_alg} - Algebraically calculated Standard Deviation')

    xplot = np.linspace(min(X)-5*bw, max(X)+5*bw, 10_001)
    yplot = np.zeros_like(xplot)
    for x, w in zip(X, W):
        yplot += norm.pdf(xplot, x, bw)*w

    std_num = weighted_std(xplot, yplot)
    # print(f'{std_num} - Numerically calculated Standard Deviation')
    
    print(std_alg/std_num)
    # if math.isnan(std_num):
    #     print(bw)
    #     print(X)
    #     print(W)
    #     print('')

In [ ]:
%%time
# visualizing when standard deviation is greater than the mean
fig = plt.figure(figsize=(6,3))
ax = fig.add_subplot(1,1,1, projection='ternary')
t0, l0, r0 = np.random.dirichlet(np.ones(3), size=10_000).T
# stds = np.array([t0, l0, r0]).std(axis=0)

vals = np.array([10, 20, 200], ndmin=2)

W = np.array([t0, l0, r0]).T
X = np.repeat(vals, len(W), axis=0)
means = np.array(np.sum(X*W, axis=1), ndmin=2).T
stds = np.sqrt(np.sum(W*(X-means)**2, axis=1))

ax.set_tlabel(vals.flatten()[0])
ax.set_llabel(vals.flatten()[1])
ax.set_rlabel(vals.flatten()[2])

pc = ax.scatter(t0, l0, r0, s=1, alpha=1, marker='.', c=stds)

cax = ax.inset_axes([1.05, 0.1, 0.05, 0.9], transform=ax.transAxes)
colorbar = fig.colorbar(pc, cax=cax)
colorbar.set_label("Standard Deviation", rotation=270, va="baseline")

ticks = [0,1]
labels = ['0', '1']
fontsize = 6
ax.taxis.set_ticks(ticks, labels=labels, fontsize=fontsize)
ax.laxis.set_ticks(ticks, labels=labels, fontsize=fontsize)
ax.raxis.set_ticks(ticks, labels=labels, fontsize=fontsize)

print(np.std(vals))

# def kl2() before I got rid of alpha

In [ ]:
# def kl2(data,
#         xplot,
#         group_constraints=None,
#         alpha=None,
#         BW_factors=None, 
#         evtargets=None,
#         represented=None,
#         nruns=1000,
#         progressbar=True
#        ):
#     # group_constraints=None
#     # alpha=None
#     # BW_factors=None
#     # evtargets=None
#     # represented=None
#     # nruns=1000
#     # progressbar=True
#     """
#     This function performs advanced KDE to a dataset, accommodating 
#     multiple sources of uncertainty and conveying a series of possible 
#     distributions as part of a Dirichlet Process.

#     INPUTS:
#         data                All data points considered for this analysis in array-like format
#         xplot               This must be a user input so you know how to plot the resulting yplot. Use np.linspace(start, stop, num)
#         group_constraints   This is a dictionary where the key is a tuple of indices and the value is the proportion of weight those indices should take up
#         alpha               Array of weights for each point in "data", used as input in the Dirichlet function. Values must be >1.
#         BW_factors          Bandwidth factors for each point in "data". Mean will be taken as one
#         evtargets           The set of feasible targets for expected value of the resulting plot. These values are used to create a KDE plot, which is sampled from for each iteration's target expected value
#         represented         What percentage of the overall data is represented by this dataset. Weight represented by new, random values
#         nruns               How many probability density functions are produced. Recommended minimum 1000, which is the autofill value
#         progressbar         Boolean that dictates whether messages are printed or not. This should be True unless you're running many separate simulations in a row.

#     OUTPUTS:
#         xplot               Array of x values produced with np.linspace(). May have been resized to accommodate high values
#         yplot               Array of (xplot * nruns) representing all resulting probability density functions
#         dict_analytics      Dictionary of some results, including all_x, all_w, and all_ev
#     """
#     ####################################################################################
#     ######################  CHECK INPUT DATA ###########################################
#     ####################################################################################
#     X = np.array(data).flatten()
#     lst1 = []
#     lst2 = []
#     messages = []
#     #check group_constraints and represented
#     if group_constraints is None:
#         if represented is None:
#             represented = 0.6
#             messages.append('represented has defaulted to 0.6, indicating that this dataset represents 60% of all values')
#         elif represented > 1 or represented < 0:
#             raise ValueError("The variable 'represented' must be between 0 and 1")        
#         group_constraints = {
#             tuple(np.arange(len(X))): represented,
#         }
#         messages.append('No group constraints were provided')
#     else:
#         # check for repeating indices
#         gc_inds = [item for lst in group_constraints.keys() for item in lst]
#         repeats = set([x for x in gc_inds if gc_inds.count(x) > 1])
#         if len(repeats)>0:
#             raise ValueError(f'The following indices in group_constraints are repeating: {repeats}')

#         # check for out of bounds indices
#         oob = [ele for ele in gc_inds if ele>=len(data)]
#         if len(oob)>0:
#             raise ValueError(f'group_constraints indices out of bounds. Dataset length: {len(X)}. Indices: {oob}')

#         # check for unused indices
#         ind_remainder = [ele for ele in np.arange(len(X)) if ele not in gc_inds]
#         gc_sum = sum(group_constraints.values())

#         # check if represented and group_constraints agree - allocate weight as needed.
#         if len(ind_remainder) > 0: #there are remaining indices
#             if represented is None:
#                 raise ValueError(f'Indices were left out of group_constraints and represented was not given. One of these must be corrected. Indices include {ind_remainder}')
#             elif represented > 1 or represented < 0:
#                 raise ValueError("The variable 'represented' must be between 0 and 1")    
#             elif represented < gc_sum:
#                 raise ValueError(f'represented is less than the sum of group_constraints, so there is no remaining weight to be allocated to unconstrained indices: {ind_remainder}')
#             group_constraints[tuple(ind_remainder)] = represented-gc_sum

#         else: #there are no more remaining indices
#             if represented is None:
#                 represented = gc_sum
#                 messages.append(f'represented defaulted to the sum of group_constraints: {gc_sum}')
#             elif represented > 1 or represented < 0:
#                 raise ValueError("The variable 'represented' must be between 0 and 1")    
#             elif represented != gc_sum:
#                 messages.append(f'represented does not equal the sum of group_constraints and all indices are accounted for, so represented is being changed from {represented} to {gc_sum}')
#                 represented = gc_sum
    
#     gc_sum = sum(group_constraints.values())
#     if represented != gc_sum:
#         raise ValueError(f'represented does not equal the sum of group constraints. represented={represented}, sum(gc)={gc_sum}')
#     elif represented <= 0 or represented > 1.0:
#         raise ValueError(f'represented must be greater than zero and less than or equal to one. Input was {represented}')
        
#     #check alpha
#     if alpha is None:
#         alpha = np.zeros_like(X).astype(float)
#         for group, perc in group_constraints.items():
#             for ind in group:
#                 alpha[ind] += perc/len(group)
#         alpha /= min(alpha)
#         messages.append('alpha has defaulted to uniform for all data points.')
#     else:
#         alpha = check_array(alpha, X, 'alpha')
#         if min(alpha) < 1:
#             raise ValueError('alpha values must be at least 1.0')

#     #check BW_factors
#     wstd = alpha/sum(alpha)
#     base_ev = sum(X*wstd)/sum(wstd)
#     # base_std = np.sqrt(sum(wstd*(X-sum(X*wstd)/sum(wstd))**2)/sum(wstd))    
#     # iqr = weighted_quantile(X, wstd, 0.75, output='perc2val') - weighted_quantile(X, wstd, 0.25, output='perc2val')
#     # bw_base = 0.9 * min([base_std, iqr/1.34]) * len(X)**-0.2
#     bw_base = weighted_bw(X, wstd, bw_method='silverman')
#     if BW_factors is None:
#         # BW_factors = np.ones(len(X)+1)
#         BW_factors = np.ones(len(X))
#         messages.append('BW_factors has defaulted an array of ones so all bandwidths will be equal.')

#     else:
#         BW_factors = check_array(BW_factors, X, 'BW_factors')
#         BW_factors = BW_factors / np.mean(BW_factors)
#         # BW_factors = np.concatenate([BW_factors, [1]])

#     #check evtargets
#     if evtargets is None:
#         evtargets = np.random.normal(base_ev, bw_base, nruns)
#         messages.append(f'evtargets has defaulted to a normal distribution with mean = expected value, stdev = bandwidth calculated with the Silverman method')
#     else:
#         evtargets = np.array(evtargets).flatten()        

#     #check xplot - must be custom input array, likely from np.linspace(low, high, number_of_steps)
#     if len(xplot) == nruns:
#         nruns += 1
#         messages.append('nruns has been increased by 1 to avoid being the same integer as xplot. This is to avoid confusing dimensions of the results.')


#     if progressbar:
#         for message in messages:
#             print(message)
            
            
#     ####################################################################################
#     ######################  FUNCTION STARTS HERE #######################################
#     ####################################################################################

#     #collect metadata
#     n_added = 1
#     W_all = np.zeros(shape=(nruns,len(X)+n_added))
#     X_all = np.zeros(shape=(nruns,len(X)+n_added))
#     EV_all = np.zeros_like(range(nruns))
#     yplot = np.zeros_like(xplot, shape=(nruns,len(xplot)))
#     xplot_bust = []
        
#     #run Monte Carlo simulations
#     for run in tqdm(range(nruns), desc='FOR LOOP PROGRESS', disable=not progressbar):
#         # reset data
#         X = np.array(data).flatten()
        
#         # pull target
#         target = evtargets[run % len(evtargets)]

#         #adjust alpha for each group_constraint, then select weight from Dirichlet distribution
#         W = np.zeros_like(X).astype(float)
#         for ind, perc in group_constraints.items():
#             alpha_group = alpha[list(ind)]
#             # need to make sure minimum alpha is one while maintaining relative weight ratios
#             alpha_group = alpha_group / np.min(alpha_group)
#             wnew = np.random.dirichlet(alpha_group)*perc
#             for i, w in zip(ind, wnew):
#                 W[i] = w.copy()

#         # simulate phantom kernel
#         EV = sum(X*W)/sum(W)
#         if represented == 1:
#             xnew = np.array(target).flatten()
#             wnew = np.array(0).flatten()
#         else:
#             xnew = target + (target-EV)*(represented)/(1-represented)
#             xnew = np.array(np.max([xnew, 0])).flatten()
#             wnew = np.array(1-represented).flatten()

#         # calculate bandwidth such that variance of the resulting plot >= variance of the original plot
#         BW1 = bw_dirichlet(X, alpha, BW_factors, Wrun=W, bw_method='silverman')
#         BW1 = np.concatenate([BW1, [np.mean(BW1)]])
#         X = np.concatenate([np.array(X).flatten(), xnew])
#         W = np.concatenate([np.array(W).flatten(), wnew])
#         alpha_phantom = np.sum(alpha)*(1-represented)/represented
#         BW2 = bw_dirichlet(X, np.concatenate([alpha, [alpha_phantom]]), np.concatenate([BW_factors, [1.]]), Wrun=W, bw_method='silverman')
#         BW = np.max([BW1, BW2], axis=0)
        
#         # calculate bandwidth with baseline bandwidth
#         # BW = BW_factors*bw_base

#         # re calculate bandwidth each time
#         # bw_basenew = weighted_bw(X, W, bw_method='silverman')
#         # BW = BW_factors*bw_basenew
        
#         # record results        
#         W_all[run] = W
#         X_all[run] = X
#         EV_all[run] = sum(X*W)/sum(W)

#         #check to see if xplot captures everything
#         hi = np.max(X + 3*BW)
#         if max(xplot) < hi:
#             xplot_bust.append(hi)
#             # xplot = np.linspace(0,hi,len(xplot))
#             # if progressbar:
#                 # print(f'The upper limit of xplot has been increased to accommodate the upper limit: {hi}')
#                 # print(f'Consider adjusting the upper limit of xplot. np.max(X + 3*BW) is {np.max(X + 3*BW)} and np.max(xplot) is {np.max(xplot)}')
        
#         # create each PDF and ensure it's valid and sums to 1
#         yi = norm.pdf(xplot.reshape(-1,1), X, BW)*W
#         # yplot[run] = sum(yi.T)
#         yplot[run] = yi.sum(axis=1)
#         yplot[run] /= np.trapz(yplot[run], xplot)

#     dict_analytics = {}
#     dict_analytics['all_x'] = X_all
#     dict_analytics['all_w'] = W_all
#     dict_analytics['all_ev'] = EV_all
#     if len(lst1) > 0:
#         dict_analytics['lst1'] = lst1
#     if len(lst2) > 0:
#         dict_analytics['lst2'] = lst2
#     if len(xplot_bust) > 0 and progressbar:
#         print(f'X+3*BW was greater than max(xplot)={np.max(xplot)} for {len(xplot_bust)}/{nruns} runs, with a max of {np.max(xplot_bust)}')


#     return xplot, yplot.T, dict_analytics